# QEC The Basics of Classical and Quantum Error Correction

$
\renewcommand{\ket}[1]{|{#1}\rangle}
\renewcommand{\bra}[1]{\langle{#1}|}
$

Execute the cells below to load all the necessary packages for this lab.

In [ ]:
!pip install cudaq
!pip install cudaq_qec
!sudo apt-get update && sudo apt-get install -y gfortran
!pip install cudaq-solvers 

In [ ]:
# install `qutip` and `ipywidgets` in the current Python kernel. Skip this if they are already installed.
# `matplotlib` is required for all visualization tasks.
# Make sure to restart your kernel if you execute this!
# In a Jupyter notebook, go to the menu bar > Kernel > Restart Kernel.
# In VSCode, click on the Restart button in the Jupyter toolbar.

# The '\' before the '>' operator is so that the shell does not misunderstand
# the '>' qualifier for the bash pipe operation.

import sys

try:
    import matplotlib.pyplot as plt
    import qutip
    import ipywidgets as widgets
    import matplotlib_venn

except ImportError:
    print("Tools not found, installing. Please restart your kernel after this is done.")
    !{sys.executable} -m pip install qutip\>=5 matplotlib\>=3.5 matplotlib_venn
    !{sys.executable} -m pip install ipywidgets
    print("\nNew libraries have been installed. Please restart your kernel!")

In [ ]:
import cudaq
from cudaq import spin
from cudaq.qis import *
import numpy as np
import random
import matplotlib.pyplot as plt


from typing import List
import ipywidgets as widgets
from ipywidgets import interact, Output, VBox, HBox
from IPython.display import display

## 1.1 The Basics of Error Correction

Classical [Error correction](https://en.wikipedia.org/wiki/Error_correction_code) (EC) is the practice of redundantly encoding data using additional bits such that if some of the data bits are corrupted (flipped from a 0 to a 1 or vice versa) by external noise, the error can be fixed and the integrity of the original data preserved. There are many potential ways to accomplish this and any specific procedure is referred to as an EC code.

It is thanks to EC that you can still watch a DVD that has many scratches, enjoy a clear telephone conversation despite the signal becoming noisy after transmission over long distances, or scan a QR code at a restaurant that is partially obscured. Try it yourself. Pull up a QR code and try covering different parts with your hand and notice that your phone can still interpret the link despite obfuscation of some parts of the grid. EC has its limits, but in many cases can essentially eliminate any negative impacts of noise.

There are five aspects to any general EC procedure:

1. All EC procedures assume there is some initial information that needs to be preserved or transmitted. The sender and recipient need to interact with the same information even if it is stored in different ways throughout an EC procedure.

2.  An EC procedure first **encodes** the information across $n$ data bits such that $n$ is larger than the minimum number of bits necessary to store the information. For example, the repitition code that we'll cover in the next section uses 3 bits to encode some binary information stored on a single bit by simply repeating the information stored on the single bit three times.

3. The encoding procedure produces **logical codewords** which are the redundant encodings that map to the logical states.  For example, logical 1 could be defined with the codeword 111.

4. A logical codeword then proceeds through a **noisy channel**.  A noisy channel has some probability of randomly corrupting (flipping) any of the data bits.

5.  The recipient then receives the encoded data and needs to decode it to determine if an error occurred and how they might fix it. Another way to say this, is that the **decoder** takes the message the recipient receives and determines which logical codeword is "closest" to it. If the decoder produces the correct logical codeword, the EC procedure worked. If not, a **logical error** occurred and the recipient incorrectly interprets the message - the worst case scenario.

No EC procedure is perfect, and is usually benchmarked against **[Shannon's limit](https://en.wikipedia.org/wiki/Noisy-channel_coding_theorem)** which is the theoretical limit of the rate of noise free data transfer that can occur through a channel with a given bandwidth and noise level.

In practice, and EC code need to be just "good enough" to ensure that errors do not usually impede the target application. Generally speaking, the more redundancy (extra bits) used in the encoding process, the better the EC will be, but this also comes at the cost of more memory and time to perform more sophisticated encoding and decoding. Thus, there is always a tension between the resources available for error correction and the logical error rate of the procedure necessary for a given application.

## 1.2 The Repetition Code

The most basic EC code is called the repetition code.

Consider encoding the information in a single bit (0 or 1). The repetition code simply adds more bits which are in the same state. So a 3-bit repetition code encodes the logical 0 state ($0_L$) as 000 and the logical 1 state ($1_L$) as 111, making 000 and 111 the logical codewords.

Now, assume $0_L$ is transmitted through a noisy channel such that each data qubit has a probability p=0.1 of flipping erroneously. This means there are eight possible states the encoded message could be in after proceeding through the noisy channel. The states can be sorted into two groups. The space of all logical codewords (000 and 111) is called the **codespace**:

| Codespace (as bitstrings)    | Codespace (as logical states)|
| ----------- | ----------- |
| 000 | $0_L$ |
| 111 | $1_L$ |


while the rest of all the potentially received messages belong to the **error space** as shown in the table below.

The job of the decoder is to map encodings in the error space back to a logical codeword in the codespace.  This is usually done through the help of a calculated property that describes the state called a **syndrome**.  In this case, the syndrome is a majority count of the bits. So, the syndrome of the 101 state would be 2 (the count of how many 1's) and corresponds to logical 1 and the syndrome of 001 would be 1.  Additionally, the syndrome of 111 is 3 and the syndrome of 000 os 0.  So we can apply the decoding rule that a message with a syndrome of 2 or 3 would be decoded as 111, and message with a syndrome of 0 or 1 would be decoded as 000.

| Error Space   | Closest logical state to the received message | Error, if only one error occurred in transmission | Syndrome |
| ----------- | ----------- | ----------- | ----------- |
| 001 | 000 | right most bit | 1|
| 010 | 000 | middle bit | 1|
| 100 | 000 | left most bit | 1 |
| 110 | 111 | right most bit | 2|
| 101 | 111 | middle bit |  2 |
| 011 | 111 |left most bit | 2 |

Any single bit error can be correctly identified and corrected, but two or more bit flips will result in a logical error.  There is no way to know for certain if 110, for example, is a single error from the $1_L$ codeword or two bit errors from the $0_L$ codeword. However, the repetition code still greatly reduces the probability of a logical error compared to no encoding. To be very conservative you could discard results where any error was detected and resend the message.  In this error detection (not correction) case,  a logical error will only occur in the highly unlikely case of three bit flip errors, where 000 was transmitted but 111 was received.  This approach requires additional data transfers to compensate for the cases where errors occurred. Error correction can eliminate the need for extra data transfers at the expense of possibly mistaking a 2-bit error for a 1-bit error.  This hints at an important property of error correction codes: distance.

Codes are characterized with the so called [n,k,d] notation where n is the number of data bits used to encode k logical bits and d is the **distance**. The distance is the number of errors that must occur for one logical codeword to become the next closest codeword. In our example of the three bit repetition code $n=3$, $k = 1$, and $d=3$, so we refer to this as a [3,1,3] code. The number of errors a code can correct $t$ is a function of code distance and can be calculated as $t = \text{floor}[(d-1)/2]$

The table below shows the likelihood of the four possible scenarios below. Notice the three bit repetition code with majority count will transmit the message with 0.972 probability of success, a significant improvement over the original probability of 0.9.  The **logical error rate** is equal to $1-p$, where $p$ is the probability of success.  In the case of the 3-bit repetition code, the logical error rate is 0.028.

<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/errortable.png?raw=1" alt="Drawing" style="width: 600px;"/>





In [ ]:
### Exercise  1 - The repetition code:

You now know enough to code up the repetition code. The exercise below will require you to generalize the repetition code so it will work with $n$ bits. Fill in the #TODO sections and then observe the plots that are generated. What conclusions can you draw from the code performance using more bits?  What do you notice about the logical error rate relative to the physical error rate


In [ ]:
def encode(bit, n):
    """Function that encodes a single bit rendundantly n times

    Parameters
    ----------
    bit: int
        Input bit (1 or 0)
    n : int
        repetitions to use for encoding

    Returns
    -------
    str
        string of length n redundantly encoding bit
    """
    #TODO



def decode(bits):
    """Function that decodes a message using majority voting to determine the closest codeword

    Parameters
    ----------
    bits: str
        bitstring corresponding to message that has passed through noisy channel

    Returns
    -------
    int
        1 or 0 corresponding to decoded codeword
    """
    #TODO



def transmit(bits, p_error):
    """Function that receives a codeword, and randomly flips each bit with probability p_error to emulate transmission through noisy channel

    Parameters
    ----------
    bits: str
        bitstring corresponding to an encoded message without noise
    p_error: float
        probability that a bit will flip through transmission

    Returns
    -------
    int
        1 or 0 corresponding to decoded codeword
    """
    #TODO


def simulate_logical_error_rate(n, p_error, trials):
    """Function to determine the logical error rate of an n-bit repetition code over specified number of trials.

    Parameters
    ----------
    n: int
        specifies n-bit repetition code to use
    p_error: float
        probability that a bit will flip through transmission
    trials: int
        number of trials used to determine logical error rate

    Returns
    -------
    float
        The logical error rate `n_errors/trials`
    """
#TODO



def plot_logical_vs_physical_error_rate(n, trials):
    """Function to plot logical vs physical error rate for fixed n and number of trials.

    Parameters
    ----------
    n: int
        specifies n-bit repetition code to use
    trials: int
        number of trials used to determine logical error rate
    """

    #TODO

    plt.figure(figsize=(10, 6))
    plt.plot(p_values, logical_error_rates, marker='o')
    plt.title('Logical Error Rate vs Physical Error Rate')
    plt.xlabel('Physical Error Rate')
    plt.ylabel('Logical Error Rate')
    plt.grid(True)
    plt.show()

# Plot 2: Logical Error Rate vs n
def plot_logical_vs_repetitions(p_error, max_n, trials):
    """Function to plot logical error rate vs bits used for redundant encoding

    Parameters
    ----------
    max_n: int
        specifies the maximum n-bit repetition code to use
    p_error: float
        probability that a bit will flip through transmission
    trials: int
        number of trials used to determine logical error rate
    """

    #TODO
    plt.figure(figsize=(10, 6))
    plt.plot(n_values, logical_error_rates, marker='o')
    plt.title('Logical Error Rate vs n')
    plt.xlabel('n')
    plt.ylabel('Logical Error Rate')
    plt.grid(True)
    plt.show()

# Example Usage
n = 3         # Number of repetitions for the first plot
p_error = 0.1       # Physical error rate for the second plot
trials = 10000

# Generate the plots
plot_logical_vs_physical_error_rate(n, trials)

max_n = 20
plot_logical_vs_repetitions(p_error, max_n, trials)

## 1.3 More Efficient EC Codes (The Hamming Code)

There are many clever ways to improve the efficiency of EC codes. One common way is to make use of a concept called **parity checks**. Parity checks provide a clever way to index where errors occur, without a brute force statistical approach like the repetition code.

[The Hamming code](https://en.wikipedia.org/wiki/Hamming_code) is a great example of a parity check code.  The [7,4,3] Hamming code consists of four data bits ($d_1, d_2, d_3, d_4$) and three additional parity check bits ($p_1, p_2, p_3$). This example will only consider single bit errors as this is another distance 3 code and can only correct single bit errors.

This is accomplished by each parity bit encoding a parity, or the mod2 sum of a subset of the data bits.  The [Venn diagram](https://en.wikipedia.org/wiki/Hamming_code) below depicts the encoding.  In this example, $p_1$ encodes the parity of $d_1$, $d_2$, and $d_4$.  If our data bits ($d_1d_2d_3d_4$) were 0110, then $p_1$ would be calculated to be 1.

<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/Hamming(7,4).svg?raw=1" alt="Drawing" style="width: 300px;"/>

Either using the static Venn diagram above or the interactive one generated by executing the cell below,
reason through the following example:

> If you wanted to send the message 0110 (here $d_1 = 0$, $d_2 = 1$, $d_3 = 1$, and $d_4 = 0$), appending the three parity bits to the end of the original bitstring would produce the logical codeword: 0110110 (where $p_1 = 1$, $p_2 = 1$, and $p_3 = 0$).  Note, this is a slight deviation from the traditional placement of the bits in the Hamming code done for simplicity.
>
>Errors could occur on any of the data or parity bits. Assume an error occurs on $d_2$ and the recipient receives 0010110. To produce the syndrome, the recipient can take the received data bits, 0010, and compute the expected parity. This is then compared to the parity that was sent, 110. The parity bits that disagree flag an error.
>
>In this case, the received message has parity bits 011 which disagrees with 110. Here, $p_1$ and $p_3$ are flagged.  This syndrome can only correspond to an error on $d_2$ based on the Venn diagram.  



In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from ipywidgets import VBox, HBox

# Function to calculate parity bits
def calculate_parity_bits(data_bits):
    d1, d2, d3, d4 = data_bits
    p1 = d1 ^ d2 ^ d4
    p2 = d1 ^ d3 ^ d4
    p3 = d2 ^ d3 ^ d4
    return [p1, p2, p3]

# Function to update the Venn diagram labels based on data bits
def update_venn_labels(data_bits):
    # Clear the previous output
    clear_output(wait=True)

    # Clear the computed parity bit outputs
    output_p1.clear_output()
    output_p2.clear_output()
    output_p3.clear_output()

    # Display the widgets again
    display(VBox([title, data_bits_widget, HBox([button_p1, button_p2, button_p3], layout=widgets.Layout(justify_content='space-between')), HBox([output_p1, output_p2, output_p3])]))

    # Create the Venn diagram
    plt.figure(figsize=(8, 8))
    venn = venn3(subsets=(1, 1, 1, 1, 1, 1, 1), set_labels=('p1', 'p2', 'p3'))

    # Set colors for the circles using NVIDIA color palette
    venn.get_patch_by_id('100').set_color('#76B900')  # Green
    venn.get_patch_by_id('010').set_color('#7A1FA2')  # Purple
    venn.get_patch_by_id('001').set_color('#F9A825')  # Yellow

    # Set colors for the intersections
    venn.get_patch_by_id('110').set_color('#A3A3A3')  # Light Gray
    venn.get_patch_by_id('101').set_color('#A3A3A3')  # Light Gray
    venn.get_patch_by_id('011').set_color('#A3A3A3')  # Light Gray
    venn.get_patch_by_id('111').set_color('#A3A3A3')  # Light Gray

    # Set transparency for the circles
    venn.get_patch_by_id('100').set_alpha(0.5)
    venn.get_patch_by_id('010').set_alpha(0.5)
    venn.get_patch_by_id('001').set_alpha(0.5)

    # Label the intersections with data bits
    venn.get_label_by_id('100').set_text(f'')
    venn.get_label_by_id('010').set_text(f'')
    venn.get_label_by_id('001').set_text(f'')
    venn.get_label_by_id('110').set_text(f'd1={data_bits[0]}')
    venn.get_label_by_id('101').set_text(f'd2={data_bits[1]}')
    venn.get_label_by_id('011').set_text(f'd3={data_bits[2]}')
    venn.get_label_by_id('111').set_text(f'd4={data_bits[3]}')

    plt.show()

# Function to update the Venn diagram and display the messages
def update_venn(data_bits, parity_bit):
    parity_bits = calculate_parity_bits(data_bits)

    # Clear the previous output
    clear_output(wait=True)

    # Display the widgets again
    display(VBox([title, data_bits_widget, HBox([button_p1, button_p2, button_p3], layout=widgets.Layout(justify_content='space-between')), HBox([output_p1, output_p2, output_p3])]))

    # Create the Venn diagram
    plt.figure(figsize=(8, 8))
    venn = venn3(subsets=(1, 1, 1, 1, 1, 1, 1), set_labels=('p1', 'p2', 'p3'))

    # Set colors for the circles using NVIDIA color palette
    venn.get_patch_by_id('100').set_color('#76B900')  # Green
    venn.get_patch_by_id('010').set_color('#7A1FA2')  # Purple
    venn.get_patch_by_id('001').set_color('#F9A825')  # Yellow

    # Set colors for the intersections
    venn.get_patch_by_id('110').set_color('#A3A3A3')  # Light Gray
    venn.get_patch_by_id('101').set_color('#A3A3A3')  # Light Gray
    venn.get_patch_by_id('011').set_color('#A3A3A3')  # Light Gray
    venn.get_patch_by_id('111').set_color('#A3A3A3')  # Light Gray


    # Set transparency for the circles
    venn.get_patch_by_id('100').set_alpha(0.5)
    venn.get_patch_by_id('010').set_alpha(0.5)
    venn.get_patch_by_id('001').set_alpha(0.5)

    # Label the intersections with data bits
    venn.get_label_by_id('100').set_text(f'')
    venn.get_label_by_id('010').set_text(f'')
    venn.get_label_by_id('001').set_text(f'')
    venn.get_label_by_id('110').set_text(f'd1={data_bits[0]}')
    venn.get_label_by_id('101').set_text(f'd2={data_bits[1]}')
    venn.get_label_by_id('011').set_text(f'd3={data_bits[2]}')
    venn.get_label_by_id('111').set_text(f'd4={data_bits[3]}')

    # Highlight the selected parity bit and relevant data bits
    if parity_bit == 'p1':
        venn.get_patch_by_id('100').set_edgecolor('black')
        venn.get_patch_by_id('100').set_linewidth(5)
        venn.get_patch_by_id('110').set_color('#76B900')  # Green
        venn.get_patch_by_id('101').set_color('#76B900')  # Green
        venn.get_patch_by_id('111').set_color('#76B900')  # Green
        venn.get_patch_by_id('110').set_edgecolor('black')
        venn.get_patch_by_id('110').set_linewidth(5)
        venn.get_patch_by_id('101').set_edgecolor('black')
        venn.get_patch_by_id('101').set_linewidth(5)
        venn.get_patch_by_id('111').set_edgecolor('black')
        venn.get_patch_by_id('111').set_linewidth(5)
        output_p1.clear_output()
        with output_p1:
            display(HTML(f"<b>p1 = d1 + d2 + d4 (mod 2)=</b> {data_bits[0]} + {data_bits[1]} + {data_bits[3]} (mod 2) = {parity_bits[0]}"))
    elif parity_bit == 'p2':
        venn.get_patch_by_id('010').set_edgecolor('black')
        venn.get_patch_by_id('010').set_linewidth(5)
        venn.get_patch_by_id('110').set_color('#7A1FA2')  # Purple
        venn.get_patch_by_id('011').set_color('#7A1FA2')  # Purple
        venn.get_patch_by_id('111').set_color('#7A1FA2')  # Purple
        venn.get_patch_by_id('110').set_edgecolor('black')
        venn.get_patch_by_id('110').set_linewidth(5)
        venn.get_patch_by_id('011').set_edgecolor('black')
        venn.get_patch_by_id('011').set_linewidth(5)
        venn.get_patch_by_id('111').set_edgecolor('black')
        venn.get_patch_by_id('111').set_linewidth(5)
        output_p2.clear_output()
        with output_p2:
            display(HTML(f"<b>p2 = d1 + d3 + d4 (mod 2)=</b> {data_bits[0]} + {data_bits[2]} + {data_bits[3]} (mod 2) = {parity_bits[1]}"))
    elif parity_bit == 'p3':
        venn.get_patch_by_id('001').set_edgecolor('black')
        venn.get_patch_by_id('001').set_linewidth(5)
        venn.get_patch_by_id('101').set_color('#F9A825')  # Yellow
        venn.get_patch_by_id('011').set_color('#F9A825')  # Yellow
        venn.get_patch_by_id('111').set_color('#F9A825')  # Yellow
        venn.get_patch_by_id('101').set_edgecolor('black')
        venn.get_patch_by_id('101').set_linewidth(5)
        venn.get_patch_by_id('011').set_edgecolor('black')
        venn.get_patch_by_id('011').set_linewidth(5)
        venn.get_patch_by_id('111').set_edgecolor('black')
        venn.get_patch_by_id('111').set_linewidth(5)
        output_p3.clear_output()
        with output_p3:
            display(HTML(f"<b>p3 = d2 + d3 + d4 (mod 2)=</b> {data_bits[1]} + {data_bits[2]} + {data_bits[3]} (mod 2) = {parity_bits[2]}"))

    plt.show()

# Create a title widget
title = widgets.Label(value="Hamming Code Visualization: Computing parity bits (p1, p2, p3)")


# Create widgets for user input
data_bits_widget = widgets.Dropdown(
    options=['0000', '0001', '0010', '0011', '0100', '0101', '0110', '0111', '1000', '1001', '1010', '1011', '1100', '1101', '1110', '1111'],
    value='1001',
    description='Data Bits (d1, d2, d3, d4):', style={'description_width': 'initial'}
)

# Create buttons for parity bits
button_p1 = widgets.Button(description='Compute p1', layout=widgets.Layout(width='150px'), style=widgets.ButtonStyle(button_color='#BBE07F'))  # Green
button_p2 = widgets.Button(description='Compute p2', layout=widgets.Layout(width='150px'), style=widgets.ButtonStyle(button_color='#BD8FD1'))  # Purple
button_p3 = widgets.Button(description='Compute p3', layout=widgets.Layout(width='150px'), style=widgets.ButtonStyle(button_color='#FCD492'))  # Yellow

# Create output areas for parity bit results
output_p1 = widgets.Output(layout=widgets.Layout(width='300px'))
output_p2 = widgets.Output(layout=widgets.Layout(width='300px'))
output_p3 = widgets.Output(layout=widgets.Layout(width='300px'))

# Define the button click events
def on_button_p1_click(b):
    data_bits_list = [int(bit) for bit in data_bits_widget.value]
    update_venn(data_bits_list, 'p1')

def on_button_p2_click(b):
    data_bits_list = [int(bit) for bit in data_bits_widget.value]
    update_venn(data_bits_list, 'p2')

def on_button_p3_click(b):
    data_bits_list = [int(bit) for bit in data_bits_widget.value]
    update_venn(data_bits_list, 'p3')

button_p1.on_click(on_button_p1_click)
button_p2.on_click(on_button_p2_click)
button_p3.on_click(on_button_p3_click)

# Define the dropdown change event
def on_data_bits_change(change):
    data_bits_list = [int(bit) for bit in change['new']]
    update_venn_labels(data_bits_list)

data_bits_widget.observe(on_data_bits_change, names='value')

# Display the widgets
display(VBox([title, data_bits_widget, HBox([button_p1, button_p2, button_p3], layout=widgets.Layout(justify_content='space-between')), HBox([output_p1, output_p2, output_p3])]))

# Initial update of the Venn diagram labels
update_venn_labels([int(bit) for bit in data_bits_widget.value])


The Hamming code takes advantage of the fact that the parity bits can encode up to $2^3 = 8$ syndromes, more than enough to consider the seven possible single bit flip errors that could occur. This means, 4 bits can be encoded with 7 bits which is an improvement over the $n$ to 1 encoding of the repetition code.

**Checkpoint:**  Suppose you sent the logical code word 0110110, but the recipient received the message 0110100.  We'll assume that at most only one error occurred.  Would the recipient be able to identify if there was an error?  If so, could they locate the error?  
Hint: errors could occur on any of the data or the parity bits.

In practice, a large message can be broken into blocks with each block is encoded using the Hamming code.  The code scales much better than the repetition code.  A Hamming code can be produced for any integer $r$ greater than 1, such that the code is characterized as $[2^r-1,2^r -r -1,3]$.  So for a message of size $2^5 -5 -1 = 26$, the Hamming code would require only $2^5-1 =31$ bits while a three bit repetition code would require $26*3 = 78$ bits.

### Exercise  2 - The matrix form of the Hamming code:
    
The Hamming code is commonly constructed with special matrices so a few simple linear algebra operations can encode and decode messages. The next two cells will have you define these matrices and see if you can reproduce the example above.  


First, define the generator matrix $G$ such that a dot product between the message and $G$  mod2 performs the valid encoding. Hint: G should be a 4x7 matrix.

In [ ]:
message = np.array([0, 1, 1, 0])

# The G matrix should properly encode the message when the following calculation is performed
G = np.array([
#FILL IN G.
])

encoded = np.dot(message, G) % 2
print(encoded)

Now, define the parity check matrix $H$ such that $Hv \mod 2$ produces a syndrome, where $v$ is the received message vector.

In [ ]:
received = np.array([0, 0, 1, 0,1,1,0])
print(received)

# Define the parity check matrix H which takes a message and determines the syndrome.
H = np.array([
#FILL IN H
])

decoded = np.dot(H, recieved) % 2

# Should print [0 1 1]
print(decoded)

## 1.4 What Makes QEC so Hard?

QEC shares the same goal as classical EC, but comes with a number of unique challenges thanks to the properties of quantum mechanics.  This section will list the primary differences, and the following section will explain how these challenges can be addressed.

1.  Continuous Errors - Classical errors are always discrete bit flips. Quantum errors are continuous and can manifest in an infinite number of ways, potentially shifting a qubit's state to any point on the Bloch sphere. For instance, the figure below illustrates many possible errors that affect a qubit starting in the $\ket{0}$ state. Errors can perturb states incoherently (from environmental effects) or coherently from slight hardware imperfections. This invites the question, "Does QEC require an infinite amount of resources to correct errors?"
   
<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/c_error.png?raw=1" alt="Drawing" style="width: 300px;"/>

2. No Cloning - Quantum states cannot be copied. That is to say that the following expression holds:$~\nexists U \text{ such that } U(\ket{\psi} \otimes \ket{\rho}) =  \ket{\psi} \otimes\ket{\psi}$. This means we cannot just send multiple copies of the quantum state through the noisy channel like the classical repetition code.

3. Destructive Measurement - In classical EC, the state can be accessed at any time, making decoding much easier. Measuring a quantum state collapses it, making the EC moot if the state is destroyed. Therefore, more clever ways to extract syndromes are required. A secondary consequence of this fact is sampling error.  Even if an algorithm could perform perfectly ensuring no sources of error, many applications require statistical sampling of the resulting state. If we sampled $\ket{\psi} = \alpha\ket{0} + \beta\ket{1}$ the frequency of 0's would be close to $\alpha^2$ but deviate based on the number of samples per the Central Limit Theorem.

4. Scalability - Though scalability is an issue for classical EC, it is far more severe for QEC. Today's noisy intermediate scale quantum devices are very difficult to control, so each additional qubit required for QEC comes at great cost.  Qubits also have short coherence times, so QEC procedures must complete within strict time constraints which gets harder at scale. Finally, the threshold theorem is in play. In classical EC, adding more bits always reduces the logical error rate. This is not true for quantum - physical qubits must have noise below a specific threshold in order for scaling the code to improve the error rates,  otherwise, the results just get worse.


## 1.5 There is still hope for QEC!

The challenges discussed above are daunting but there are many ingenious techniques developed to help circumvent them. That said, practical QEC remains difficult to realize and is an extremely active research field - viewed as one of the most important prerequisites for useful quantum computing.  This section will begin to bridge the gap between classical EC and QEC.


### Digitization of errors

Errors can perturb states incoherently from environmental effects or coherently from slight hardware imperfections.  While both types of errors can be addressed, we’ll focus on coherent errors first because they’re often easier to isolate and analyze.

For instance, a rotation gate that should be at an angle of $\frac{\pi}{16} \approx 0.196 $ ends up being more like 0.17. This may seem inconsequential, but imperfections like this accumulate and quickly ruin the outcome of a quantum algorithm.  Execute the code block below and use the slider to change the number of rotation gates executed to see how the error can become substantial.  Feel free to experiment with different values for the `angle`, `noisy_angle`, and the rotation axis in the `rotation_kernel`.




In [ ]:
# Angles of rotation of a qubit
angle = np.pi / 16
noisy_angle = 0.17

# Kernel to initialize a qubit in the zero ket state and rotate it about the x axis by given angle n times
@cudaq.kernel
def rotation_kernel(n: int, angle: float):
    qubit = cudaq.qubit()
    for _ in range(n):
        rx(angle, qubit) # CHANGE THE ROTATION AXIS

# Function to plot sample results
def plot_results(results1, results2):
    # Convert the sample results to a dictionary
    result_dictionary1 = {k: v for k, v in results1.items()}
    result_dictionary2 = {k: v for k, v in results2.items()}

    # Get all unique x-values from both dictionaries
    all_keys = set(result_dictionary1.keys()).union(set(result_dictionary2.keys()))
    all_keys = sorted(all_keys)

    # Convert the dictionary to lists for x and y values
    x1 = list(all_keys)
    y1 = list(result_dictionary1.values())
    y2 = list(result_dictionary2.values())

    # Create the combined histogram
    bar_width = 0.35
    x_indices = range(len(x1))

    plt.bar(x_indices, y1, width=bar_width, color='#76B900', label='Noise-Free Results')
    plt.bar([i + bar_width for i in x_indices], y2, width=bar_width, color='#484848', label='Noisy Results')

    # Add title and labels
    plt.title('Comparing sampling results of n applications of a noise-free gate with a noisy version')
    plt.xlabel("Basis States")
    plt.ylabel("Frequency")
    plt.xticks([i + bar_width / 2 for i in x_indices], x1)
    plt.legend()

    # Show the plot
    plt.tight_layout()
    plt.show()

# Function to update the plot based on the slider value
def update_plot(num_rotations):
    expected_result = cudaq.sample(rotation_kernel, num_rotations, angle)
    noisy_result = cudaq.sample(rotation_kernel, num_rotations, noisy_angle)
    plot_results(expected_result, noisy_result)

# Create an interactive slider
slider = widgets.IntSlider(min=1, max=20, step=1, value=1, description='n:', continuous_update=False)
interact(update_plot, num_rotations=slider)

Among the various coherent errors that can occur on a qubit storing the quantum state $\ket{\psi} = \alpha \ket{0}+\beta\ket{1}$, we will focus on three specific types:
* **Bit flip errors** swap a qubit's amplitudes, transforming $\ket{\psi}$ to $\beta\ket{0}+\alpha\ket{1}$.
* **Phase flip errors** introduce a sign change in one of the amplitudes, transforming $\ket{\psi}$ to $\alpha\ket{0}-\beta\ket{1}$.
* **Combining a bit flip with a phase flip error** swaps amplitudes and applies a sign change, transforming $\ket{\psi}$ to $\beta\ket{0}-\alpha\ket{1}$.

Run the cell below to open an interactive tool that allows you to visualize the impact of different error types on various quantum states. Observe how some error types may not alter the state. Why do you think that happens? What patterns can you identify?

In [ ]:
# Execute this cell to see the interactive widget
# Don't concern yourself with the code below this line
# Function to update and display the Bloch sphere
def update_bloch_sphere(theta, phi, error_type):
    alpha = np.cos(theta / 2)
    beta = np.sin(theta / 2) * np.exp(1j * phi)
    coefficients = [complex(alpha, 0), complex(0, beta)]
    error_types = ['Bit Flip', 'Phase Flip', 'Bit & Phase Flip']
    @cudaq.kernel
    def initial_state_kernel(coefficients: list[complex]):
        qubit = cudaq.qvector(coefficients)

    @cudaq.kernel
    def initial_state_error(coefficients: list[complex], error: int):
        qubit = cudaq.qvector(coefficients)
        if error == 0 or error == 2:
            # bit flip error
            x(qubit)
        if error == 1 or error == 2:
            # phase flip error
            z(qubit)

    state_no_error = cudaq.get_state(initial_state_kernel, coefficients)
    state_with_error = cudaq.get_state(initial_state_error, coefficients, error_type)

    blochSphereList = []
    # Define a sphere object representing the state of the single qubit
    sphere = cudaq.add_to_bloch_sphere(state_no_error)
    blochSphereList.append(sphere)
    sphere = cudaq.add_to_bloch_sphere(state_with_error)
    blochSphereList.append(sphere)

    # Create output widgets for the Bloch spheres and text
    out1 = Output()
    out2 = Output()
    text1 = Output()
    text2 = Output()

    with out1:
        cudaq.show([blochSphereList[0]], nrows=1, ncols=1)
    with out2:
        cudaq.show([blochSphereList[1]], nrows=1, ncols=1)
    with text1:
        print(f"|ψ> = cos(θ/2)|0⟩ + e^(iφ)sin(θ/2)|1⟩")
    with text2:
        print("|ψ⟩ with a ", error_types[error_type], " error")

    display(VBox([HBox([VBox([text1, out1]), VBox([text2, out2])])]))

# Create the interactive widget
theta_slider = widgets.FloatSlider(value=np.pi/2, min=0, max=2*np.pi, step=0.01, description='θ (radians):')
phi_slider = widgets.FloatSlider(value=0, min=0, max=np.pi, step=0.01, description='φ (radians):')
error_selector = widgets.Dropdown(options=[('None', -1), ('Bit Flip', 0), ('Phase Flip', 1), ('Bit & Phase Flip', 2)], value=-1, description='Error Type:')

interact(update_bloch_sphere, theta=theta_slider, phi=phi_slider, error_type=error_selector)


Once we have identified one of these errors, we can correct it. For instance, if a qubit has undergone a bit flip error, we can correct it by applying an $X$ gate. Similarly, to correct a qubit that has experienced a phase flip error, we simply apply a $Z$ gate. How would you correct a qubit that has been identified as having undergone a bit flip error followed by a phase flip error?  

We can address all coherent errors with a key insight: although the Bloch sphere suggests errors can occur through infinitely many possible rotations, all such errors can be broken down into three basic forms &mdash; bit flips, phase flips, or a combination of both bit flips and phase flips.  

If you'd like an explanation of why this decomposition works, consult the optional section below. For now, remember that by detecting and correcting these three core error types, we can effectively handle any coherent noise.  

> **Optional:** Consider a qubit in the following normalized state.
> $$ \ket{\psi}  =  \cos\frac{\theta}{2}\ket{0} + e^{i\phi}\sin\frac{\theta}{2}\ket{1} $$
>
>  Coherent errors can be represented by the application of a Unitary $U(\delta\theta,\delta\phi)$ which acts on the ideal state and perturbs it.
>$$ U(\delta\theta,\delta\phi)\ket{\psi}  =  \cos\frac{\theta +\delta\theta}{2}\ket{0} + e^{i\phi+\delta\phi}\sin\frac{\theta+\delta\theta}{2}\ket{1} $$
> Using the fact that the Pauli matrices form a basis for any 2x2 unitary matrix and taking advantage of the identity $Y=iXZ$, the operation can be rewritten as
> $$ U(\delta\theta,\delta\phi) \ket{\psi} = \alpha_II\ket{\psi} +\alpha_X X\ket{\psi}+\alpha_Z Z\ket{\psi}+\alpha_{XZ}XZ\ket{\psi}   $$
> This means that any coherent error can be **digitized** into X-type bit flip errors ($X \ket{\psi} = \alpha X\ket{0} + \beta X\ket{1} = \alpha\ket{1} + \beta\ket{0}$), Z-type phase flip errors ($Z\ket{\psi} = \alpha Z\ket{0} + \beta Z\ket{1} = \alpha\ket{0} - \beta\ket{1}$), or a combination of the two (XZ). This makes correction much more tractable, as there are only three types of errors to consider.

### Syndrome Extraction

The no cloning principle means quantum states cannot be copied for QEC. We'll need a clever way to extract syndromes from the logical state that does not rely on repetition. But, how is this done without destroying the information that is being protected?

The solution involves **stabilizers** which are specially designed operators that act on a logical state without changing it, but still enable us to learn about errors by performing projective measurement of ancilla qubits.  The next notebook in this series will introduce stabilizers with more mathematical rigor, and the example in section 1.6 of this lab will provide a more concrete example of a simple stabilizer in action.

### Better QEC codes and AI solutions

Finally, overcoming the QEC scaling challenges will require breakthroughs on many fronts. Significant research efforts are targeting discovery of more efficient QEC codes that require fewer qubits.  AI is already showing great promise as a tool to help find new QEC codes, and accelerate decoding. Later notebooks will explore AI for QEC applications.


## 1.6 The Quantum Repetition Code

A quantum state cannot be cloned, but it can be redundantly encoded across additional entangled qubits. Let's start with a generic normalized qubit state $\ket{\psi}$:

$$\ket{\psi} = \alpha\ket{0} +\beta\ket{1}.$$

The 0 and 1 states can be encoded into a logical state making use of the larger 8-dimensional Hilbert space of three qubits:  

$$\ket{\psi}_L = \alpha\ket{000} +\beta\ket{111} = \alpha\ket{0}_L +\beta\ket{1}_L.$$

Note that this is *not* equivalent to $\ket{\psi} \otimes \ket{\psi} \otimes \ket{\psi}$.

Consider now the entire Hilbert space spanned by the eight basis states.  The basis states are separated into the logical codewords that make up the codespace:   

| Codespace    | Notation for the logical codewords|
| ----------- | ----------- |
| $\ket{000}$ | $\ket{0}_L$ |
|$\ket{111}$ | $\ket{1}_L$ |

and the remaining basis states make up the error space:

| Error space |
| ----------- |
| $\ket{001}$ |
| $\ket{010}$ |
| $\ket{100}$ |
| $\ket{011}$ |
| $\ket{101}$ |
| $\ket{110}$ |

Assume that the state $\ket{111}$ is transmitted through a noisy channel and becomes $\ket{011}$?  How might it be decoded to tell which logical codeword it is closest to?  Remember, you cannot simply examine the state and perform a majority count as no information about the state is accessible without some sort of measurement that induces wavefunction collapse.  

Consider the operators $Z_1Z_2$ and $Z_2Z_3$.  (Remember that $Z_n$ returns an eigenvalue of +1 if the nth qubit in a ket is a 0 and -1 if it is a 1).  
It turns out that these operators have special properties such that the eigenvalues produced when they act on any of the states in the codespace or error space can be extracted with ancilla qubits without disturbing the state.  This means, there is a way to extract parity check information just like the classical Hamming code!

Note: Operators with these "special properties" are called stabilizers and will be rigorously introduced in the next lab.

The details of this extraction process are described shortly, but the implication is that syndromes can be produced from the parity check results and used to identify corrections to the quantum repetition code without destroying the encoded state. Considering only single bit flip errors, the table below shows the possible syndromes, the corresponding errors, and the operation that can be applied to correct the error assuming $\ket{111}$ is the transmitted message.  

| $Z_1Z_2$ Syndrome   | $Z_2Z_3$ Syndrome| Encoded State | Single Bit Flip Correction
| ----------- | ----------- | ----------- | ----------- |
| 0 | 0 | $\ket{111}$ | none |
| 1 | 0 | $\ket{011}$ | $X_1$ |
| 0 | 1 | $\ket{110}$ | $X_3$ |
| 1 | 1 | $\ket{101}$ | $X_2$ |


Like the classical repetition code, there is no way to know for certain if a 10 syndrome corresponds to a single bit flip error from the $\ket{111}$ codeword or a two bit flip error from the $\ket{000}$ codeword.  However, it is always prudent to assume that the case with fewer errors is more likely.

The entire quantum circuit for the three qubit repetition code is shown below, where the ancilla qubits are used to compute the $Z_1Z_2$ and $Z_2Z_3$ syndromes.



<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/repcircuit.png?raw=1" alt="Drawing" style="width: 800px;"/>



It is helpful to explore in greater detail how the syndromes can be extracted using the ancilla qubits without disturbing the state.
First, consider the initial state of the first ancilla after application of a Hadamard gate and the encoded state after a bit flip has occurred on the first data qubit.

$$ \frac{1}{\sqrt{2}}(\ket{0} + \ket{1})\ket{011} $$

Next, a controlled $Z_1Z_2$ operation is applied to the data qubits resulting in

$$ \frac{1}{\sqrt{2}}(\ket{0}\ket{011} + \ket{1}Z_1Z_2\ket{011}), $$

followed by an application of the second Hadamard:

$$ \frac{1}{2} ((\ket{0}+\ket{1})\ket{011} + (\ket{0} -\ket{1})Z_1Z_2\ket{011})  =  \ket{0}(\frac{1+Z_1Z_2}{2})\ket{011} + \ket{1}(\frac{1-Z_1Z_2}{2})\ket{011}  .   $$

Now, if $Z_1Z_2\ket{011}$ is evaluated, the result is $-\ket{011}$ and the entire state simplifies to $\ket{1}\ket{011}$ meaning upon measurement, the first ancilla qubit will be measured as 1 with certainty and the data qubits remain undisturbed in the $\ket{011}$ state:


$$ \ket{0}(\frac{1+Z_1Z_2}{2})\ket{011} + \ket{1}(\frac{1-Z_1Z_2}{2})\ket{011}  =   \ket{0}(\frac{1+ -1}{2})\ket{011} + \ket{1}(\frac{1--1}{2})\ket{011}  =   \ket{1}\ket{011}.  $$

A similar analysis will show that the second ancilla qubit will be measured as 0 with certainty without distubring the data qubits.  Accordoing to the syndrome table,
this will trigger an application of the $X$ gate on the first qubit to correct the error.


### Exercise 3: Coding the Quantum Repetition Code

Now that you understand the quantum repetition code, try to code it using CUDA-Q. Fill in each of the steps below marked "#TODO". CUDA-Q contains a couple of features particularly helpful for building QEC workflows. First, and already completed for you, is the definition of a custom noise model which produces custom identity operations that can randomly perform bit flips on specific qubits. Second, you can measure the ancilla qubits within the kernel and use the result to perform a correction operation. 


Try to code all the steps and then sample the kernel to determine the logical error rate.



In [ ]:
import cudaq
import numpy as np

cudaq.set_target('density-matrix-cpu')


#First, create an empty noise model
noise_model = cudaq.NoiseModel()
p = 0.1

#Build a custom gate which applies the identity operation
cudaq.register_operation("custom_i", np.array([1, 0, 0, 1]))

#Add a bitflip noise channel to the custom_i gate applied to each qubit
noise_model.add_channel("custom_i", [0], cudaq.BitFlipChannel(p))
noise_model.add_channel("custom_i", [1], cudaq.BitFlipChannel(p))
noise_model.add_channel("custom_i", [2], cudaq.BitFlipChannel(p))

@cudaq.kernel
def three_qubit_repetition_code():
    """Prepares a kernel for the 3-bit quantum repetition code

    Returns
    -------
    cudaq.kernel
        Kernel for running the 3-bit quantum repetition code

    """

    # Create register for data and ancilla qubits
    # TODO

    # Initialize the logical |1> state as |111>
    # TODO

    # Apply custom_i to induce random bitflip errors
    # TODO

    # Extract Syndromes
    # TODO

    # Correct errors based on syndromes
    # TODO

# Run the kernel and observe results
# The percent of samples that are 000 corresponds to the logical error rate
result = cudaq.sample(three_qubit_repetition_code, noise_model=noise_model)
print(result)

# Stabilizers, the Shor code, and the Steane code

This lab was motivated by content from "[Quantum Error Correction: an Introductory Guide](https://arxiv.org/abs/1907.11157)" and "[Quantum Error Correction for Dummies](https://arxiv.org/abs/2304.08678)", both excellent resources we refer readers to for additional detail.  For a more technical introduction, see chapter 10 of "[Quantum Computation and Quantum Information](https://books.google.com/books?hl=en&lr=&id=-s4DEy7o-a0C&oi=fnd&pg=PR17&dq=quantum+computation+and+quantum+information&ots=NJ4KdqnzZt&sig=uKTETo5LLjWB9F_PV_zf0Sw3bvk#v=onepage&q=quantum%20computation%20and%20quantum%20information&f=false)" or the [PhD thesis](https://arxiv.org/abs/quant-ph/9705052) where the concept of stabilizer codes was introduced.



Execute the cells below to load all the necessary packages for this lab.

In [ ]:
import cudaq
from cudaq import spin
from cudaq.qis import *
import numpy as np
import matplotlib.pyplot as plt
from typing import List

## 2.1 Stabilizers and Logical Operators


An important subclass of QEC codes, known as **stabilizer codes**, use special operations called **stabilizers** to clean up errors in encoded quantum information, and thus "stabilize" the state.


An operation $s$ acting on a state $\ket{\psi}$ is said to be a stabilizer of the state if the state is a +1 eigenstate of the operation $s \ket{\psi} = +1 \ket{\psi}$. The high-level intuiton here is that if small errors have accumulated in a logically encoded state, the action of applying this stabilizer is to project the state back to a perfectly error-free state, and we measure $+1$. Sometimes larger errors occur, and we do not measure $+1$, which informs us something has gone wrong.


In lab 1, the codespace was defined by the set of basis codewords, such as $\ket{000}$ and $\ket{111}$ for the 3-qubit quantum repetition code. In that lab the codewords were provided to you for each code, but in a stabilizer code, we can equivalently define the codespace by providing the stabilizers which stabilize each basis codeword.  In practice, this process of defining a code by the stabilizers is much more efficient and scalable as the codes grow larger.

The codespace $C$ can be defined as formed by all $\ket{\psi}$ such that $s_i\ket{\psi} = +1 \ket{\psi}$ for each $s_i\in S$, where these $s_i$ are stabilizers which form a group $S$ (note: in some texts this group $S$ is called the stabilizer, not the elements). That is, the codespace is the joint +1 eigenspace fixed by the stabilizers.  

Again in lab 1, we were given the codespace and error space for the 3-qubit quantum repetition code up front. However, let's think about working backwards from the computational basis for the 3-qubit Hilbert space. These can be sorted based on the eigenvalues returned when operated on by all elements of the stabilizer group $S = \{Z_1Z_2, Z_2Z_3\}$:

| Basis state | Eigenvalue for $Z_1Z_2$ | Eigenvalue for $Z_2Z_3$ |
| ----------- | ----------- | ---------- |
| $\ket{000}$ | 1 | 1 |
| $\ket{001}$ |  1 | -1 |
| $\ket{010}$ |  -1 | -1 |
| $\ket{100}$ |  -1 | 1 |
| $\ket{011}$ |  -1 | 1 |
| $\ket{101}$ |  -1 | -1 |
| $\ket{110}$ |  1 | -1 |
| $\ket{111}$ |  1 | 1 |




The basis states that have an eigenvalue of 1 for both $Z_1Z_2$ and $Z_2Z_3$ make up the codespace $\ket{000}$ and $\ket{111}$. There are other valid stabilizers in this code, such as $Z_1 Z_3$, but any stabilizer group can be boiled down into a minimal set which can be multiplied together to generate all of the others.

This is a really powerful approach, because it eliminates the need to derive and document the basis of the codespace in advance. Instead, one can simply define an appropriate set of stabilizers to establish the codespace.

Stabilizer codes are usually characterized as $[[n,k,d]]$ (double brackets for quantum codes) where $n$ is the number of physical qubits encoding $k$ logical qubits with distance $d$. It is always the case that these codes require $n-k$ stabilizers. The reason for this is that each stabilizer splits the original $2^n$ Hilbert space in two (the +1 and -1 eigenspace), with $2^1 = 2$ degrees of freedom remaining to define the logical qubit.

The trick then becomes finding good sets of stabilizers that correspond to QEC codes with favorable properties.

### Stabilizer Properties

Three key properties for $[[n,k,d]]$ stabilizers:

1. Here we consider only to Pauli product stabilizers, that is, $s_i$ needs to be a Pauli-group element. The n-qubit Pauli group $G_n$ is a special group constructed from the Pauli matrices:

  $$ I = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}, \quad X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}, \quad Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

  The group $G_n$ consists of $4*4^n$ elements and is built by forming a group that begins with all possible length $n$ Pauli words. For example $G_3$ has  terms like $IZZ$, $ZXZ$, $YYY$, etc. The group is then closed by including the possible scalar cofficients which arise from multiplying these terms $\{ 1, -1, i, -i\}$. So $G_1$ would be

  $$G_1 \equiv \{\pm I, \pm iI,\pm Z, \pm iZ,\pm X, \pm iX,\pm Y, \pm iY\}$$


2. Each $s_i$ must be able to operate on every logical state $\ket{\psi_L}$. Furthermore, this action should leave each $\ket{\psi_L}$ fixed (i.e., the eigenvalue of $\ket{\psi_L}$ should be +1 for each logical state).

3. Stabilizers need to be measurable in any order. This means each stabilizer needs to **commute** with every other stabilizer (that is, $s_is_j = s_js_i$, or equivalently $[s_i, s_j] \equiv s_is_j - s_js_i=0$).


### Logical Operators

In addition to stabilizers, each $[[n,k,d]]$ code has $2k$ **logical operators** ($\bar{X}_i$ and $\bar{Z}_i$) which perform $X$ and $Z$ operations on the logical states. For example:

$$ \bar{Z}_2\bar{X}_1\ket{11}_L = \bar{Z}_2\ket{01}_L = -\ket{01}_L $$

These logical operators must satisfy two properties.

1. $\bar{X}_i$ and $\bar{Z}_i$ commute with all stabilizers.
2. $\bar{X}_i$ and $\bar{Z}_i$ must **anticommute** with one another if acting on the same logical qubit (i.e.,  $\{\bar{X}_i,\bar{Z}_j\}\equiv \bar{X}_i\bar{Z}_j + \bar{Z}_j\bar{X}_i= 2\delta_{ij}I$ for all $i,j$).


The next few sections will make use of stabilizers to solidify these concepts and enable coding of QEC codes far more interesting than the quantum repetition code.

## 2.2 The Steane Code

The Steane code is a famous QEC code that is the quantum version of the [7,4,3] Hamming code introduced in the first part.  One immediate difference is that the Steane code encodes a single logical qubit making it a [[7,1,3]] code.

Remember, that the Hamming code adds additional parity bits that help "triangulate" where an error occurred. In the previous exercises you constructed the generator matrix $G$ and used it to produce the logical codewords in the classical Hamming code. For example, $b=0110$ was encoded as


$$
c = bG=
 \begin{bmatrix} 0 & 1 & 1 & 0 \end{bmatrix}
\cdot
\begin{bmatrix}
1 & 0 & 0 & 0 & 1 & 1 & 0 \\
0 & 1 & 0 & 0 & 1 & 0 & 1 \\
0 & 0 & 1 & 0 & 0 & 1 & 1 \\
0 & 0 & 0 & 1 & 1 & 1 & 1
\end{bmatrix}
=
\begin{bmatrix}
  0 & 1& 1 & 0 & 1& 1& 0
\end{bmatrix}
$$

Any logically encoded state, $c$, could then be multiplied by the parity check matrix ($H$) to determine if any syndromes were triggered or not.


$$
Hc^T
=
\begin{bmatrix}
1 & 1 & 0 & 1 & 1 & 0 & 0 \\
1 & 0 & 1 & 1 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0 & 0 & 1
\end{bmatrix}
\cdot
\begin{bmatrix}
0 \\
1 \\
1 \\
0 \\
1 \\
1 \\
0
\end{bmatrix}
=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}.
$$

What was not discussed in the Hamming code section was the fact that the parity check matrix ($H$) can be used to define the codespace. A valid logical codeword is any $c$ that satisfies the relationship $Hc^T=\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}$.  As there are 7 data bits, that means there are $2^7=128$ possible encoded states between the codespace and error space.  It turns out, 16 of these fall within the codespace and 112 are in the error space. Of the 16 in the codespace, 8 have even parity (even number of 1's) while the other half has odd parity.

| Even Bitstrings in Codespace | Odd Bitstrings in Codespace |
| ----------- | ----------- |
| 0000000 | 1111111 |
| 0001111 | 1110000 |
| 0110110 | 1001001 |
| 0111001 | 1000110 |
| 1010101 | 0101010 |
| 1011010 | 0100101 |
| 1100011 | 0011100 |
| 1101100 | 0010011 |


This provides us a way to define the logical code words: $\ket{0}_L$ and $\ket{1}_L$.
The logical codewords, $\ket{0}_L$ and $\ket{1}_L$, for the Steane code are superpositions over the states corresponding to the classic even and odd codewords, respectively.



$$ \ket{0}_L = \frac{1}{\sqrt{8}}(\ket{0000000} +\ket{0001111} +\ket{0110110} + \ket{ 0111001} + \ket{1010101} + \ket{1011010}+ \ket{1100011} + \ket{1101100})  $$

$$ \ket{1}_L = \frac{1}{\sqrt{8}}(\ket{1111111} +\ket{1110000} +\ket{1001001} + \ket{1000110} + \ket{0101010} + \ket{0100101}+ \ket{0011100} + \ket{0010011})  $$


You might notice by inspection, that $\ket{0}_L = \bar{X}\ket{1}_L = X_1X_2X_3X_4X_5X_6X_7\ket{1}_L$.  That is to say, flipping all the bits swaps logical states.   Similarly, $\bar{Z} =  Z_1Z_2Z_3Z_4Z_5Z_6Z_7$ will flip the phase, transforming $\frac{1}{\sqrt{2}}(\ket{0}_L+\ket{1}_L)$ to $\frac{1}{\sqrt{2}}(\ket{0}_L-\ket{1}_L)$

The encoding circuit to produce the logical codewords is shown below, and is based off the constraints imposed by the parity check matrix.

<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/steaneencoding.png?raw=1" alt="Drawing" style="width: 500px;"/>







In [ ]:
### Exercise  4 - The Steane Code:

In the cell below, build a CUDA-Q kernel to encode the logical 0 state using the Steane code.  Sample the circuit to prove that you indeed created the appropriate superposition.  In the cells following, complete the entire Steane code by adding stabilizer checks and code to measure the logical state. Complete the numbered tasks as well to confirm your code works as expected.


In [ ]:
@cudaq.kernel
def steane_code():
    """Prepares a kernel for the Steane Code
    Returns
    -------
    cudaq.kernel
        Kernel for running the Steane code
    """

    #Initialize Registers
    #TODO

    # Create a superposition over all possible combinations of parity check bits
    #TODO

    #Entangle states to enforce constraints of parity check matrix (circuit above)
    #TODO



results = cudaq.sample(steane_code, shots_count=10000)
print(results)

The Steane code is a member of an important family of stabilizer codes known as **Calderbank-Shor-Steane (CSS)** codes. A CSS code is characterized by the property that Z and X errors can be detected and corrected independently.  A benefit of this, is that fewer ancilla qubits are required to produce the syndromes, and they can be reset and reused for each error. However, this procedure is also slower.

The stabilizers for $Z$-type errors are $S_Z = \{X_1X_2X_5X_4, X_1X_3X_4X_6, X_2X_3X_4X_7\},$ while the stabilizers for $X$-type errors are of similar form: $S_X = \{Z_1Z_2Z_5Z_4, Z_1Z_3Z_4Z_6, Z_2Z_3Z_4Z_7\}$.

Just as with the classical Hamming code, there is a nice way to visualize the syndrome results. The diagram below places each data qubit on each vertex. They are arranged such that each of the three sections or **plaquettes** corresponds to one of the stabilizers.  

The syndromes can be visually interpreted by putting a colored X on the syndromes that are flagged. Each coloring of this graph uniquely corresponds to an error on a specific qubit which is why the Steane code is often referred to as a **color code**.


<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/plaqettes.png?raw=1" alt="Drawing" style="width: 700px;"/>

You are now ready to code the rest of the Steane code.  After encoding, introduce an $X$ error and $Z$ error on the qubits of your choice.  Try performing the $X$ and $Z$ syndrome measurements using the same three ancilla qubits and resetting them in between. Make your code such that you can measure the data qubits and confirm the state of the logical qubit.  

In [ ]:
import cudaq
@cudaq.kernel
def steane_code():
    """Prepares a kernel for the Steane Code
    Returns
    -------
    cudaq.kernel
        Kernel for running the Steane code
    """

    #Initialize Registers
    #TODO

    # Create a superposition over all possible combinations of parity check bits
    #TODO

    #Entangle states to enforce constraints of parity check matrix (circuit above)
    #TODO

    #Add Errors (Optional)
    #TODO



    # Perform Stabilizer checks for Z errors
    #TODO


    # Perform Stabilizer checks for X errors
    #TODO


    # Correct X errors
    #TODO

    # Correct Z errors
    #TODO


results = cudaq.sample(steane_code, shots_count=1000)
print(results)

#Post-process Results
#TODO

Now, test your code! Just measure in the $Z$ basis as the same procedure could be performed with the $X$ basis.

1. Try adding single $X$ errors, guess which stabilizers should flag and confrm they do.
2. Add two errors. Confirm the code cannot correct the errors and a logical bitflip occurs.
3. It turns out that like the Shor code, there are alternate choices for $\bar{X}$.  Modify your counting code above and test if $X_0X_1X_4$ or $X_0X_4X_5$ are valid choices for $\bar{X}$.  

## 2.3 Steane Code Capacity Analysis with CUDA-Q QEC


[CUDA-QX](https://developer.nvidia.com/cuda-qx) is set of libraries that enable easy acceleration of quantum application development.  One of the libraries, [CUDA-Q QEC](https://nvidia.github.io/cudaqx/components/qec/introduction.html), is focused on error correction and can help expedite much of the work done above.  This final section will demonstrate how to run a code capacity memory experiment with the Steane code.

A memory experiment is a procedure to test how well a protocol can preserve quantum information. Such an experiment can help assess the quality of a QEC code but is often limited by assumptions that deviate from a realistic noise model. One such example is a code capacity experiment. A code capacity procedure determines the logical error rate of a QEC code under strict assumptions such as perfect gates or measurement.  Code capacity experiments can help put an upper bound on a procedure's threshold and is therefore a good starting place to compare new codes.

The process is outlined in the diagram below.  Assume the 0000000 bitstring is the baseline (no error).  Bitflips are then randomly introduced and produce errors in the data vector to produce results like 0100010.  If this were a real test on a physical quantum device, the data vector would not be known and a user could only proceed through the bottom path in the figure - performing syndrome extraction and then decoding the result to see if a logical flip occurred. In a code capacity experiment, the data vector with errors is known, so it can be used to directly compute if a logical state flip occurred or not.  Dividing the number of times the actual (top path) and predicted (bottom path) results agree by the total number of rounds provides an estimate of the logical error rate for the code being tested.


<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/steanecodecapacity.png?raw=1" alt="Drawing" style="width: 700px;"/>






In [ ]:
### Exercise  5 - CUDA-Q QEC Code Capacity Experiment:

CUDA-Q QEC allows researchers to streamline experiments like this with just a few lines of code.  Try running the cells below to compute the logical error rate of the Steane code under code capacity assumptions given probability of error $p$.


In [ ]:
import numpy as np
import cudaq_qec as qec

Next, load the Steane code, which is already implemented in CUDA-Q QEC.

In [ ]:
steane = qec.get_code("steane")

The parity check matrices and observables can also be extracted from the Steane code.

In [ ]:
Hz = steane.get_parity_z()
Hx = steane.get_parity_x()
H = steane.get_parity()
observable  = steane.get_observables_z()

A decoder can then be specified which takes the parity check matrix as an input.

In [ ]:
decoder = qec.get_decoder("single_error_lut", Hz)

Then, `sample_code_capacity` can be called and provided with `p`, the probability of any bit flipping, and the number of shots for the analysis.

In [ ]:
p = 0.1 # set a probability of a bit flip error occuring
nShots = 100 # specify the number of shots
syndromes, data = qec.sample_code_capacity(Hz, nShots, p)

for x in range(nShots):
    print("Data Qubits", data[x], "Syndromes", syndromes[x])

Notice how the Steane code is already defined within CUDA-Q QEC, along with a selection of decoders, and the `sample_code_capacity` API to automatically run the procedure.  Otherwise, you would need to code the entire process from scratch like you did in section 2.2 for each QEC code you want to test!

If the experiment is repeated many times with different $p$ values, a plot can be generated like the one shown below. The purple line is the $y=x$ and corresponds to the case that the logical error rate is identical to the physical error rate.  Anywhere the green line is below the purple line indicates that the Steane code was able to produce a logical error rate that is less than the physical error rate of the data qubits. When the green line is above the purple, the Steane code produced a worse logical error rate indicating that it would have been better to just use the data qubits and avoid the QEC procedure. The crossover point is an estimate for the code's threshold. Refining this estimate would require more sophisticated circuit level noise models that more accurately represent the performance of the Steane code under realistic conditions.

<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/codecapacityplot.png?raw=1" alt="Drawing" style="width: 700px;"/>

Though code capacity has much room to improve, it is a great example of the utility of CUDA-Q QEC and how simple procedures can be streamlined so users can focus on testing codes rather than coding up the details of each test.

## 2.4 The Shor Code

The first QEC code was proposed by Peter Shor in 1995, known as the [Shor code]((https://journals.aps.org/pra/abstract/10.1103/PhysRevA.52.R2493)).  The Shor code is a [[9,1,3]] code which uses 9 qubits to encode a single qubit, but can correct single $X$ or $Z$-type errors.


The motivation for the code, is that the 3-qubit repetition code can correct bit flip errors but not phase flip errors.  We can consider why this is by examining the encoded $\ket{+}_L$ state, which looks like the following:

$$ \ket{+}_L = \frac{1}{\sqrt{2}}(\ket{0}_{L~3\mathrm{bit}}+\ket{1}_{L~\mathrm{3bit}}) =  \frac{1}{\sqrt{2}}(\ket{000}+\ket{111}) $$

If a $Z_1$, $Z_2$, or a $Z_3$ error occurs, the $ \ket{+}_L$ state is transformed to $ \ket{-}_L$, another valid codeword.  This means there is no way to tell if a phase flip error occurred or not. One could produce the repetition code in the  $ \ket{+}_L$ and $ \ket{-}_L$ basis to correct $Z$ errors, but then the same problem would persist for $X$ errors.

The ingenuity behind the Shor code is to concatenate two 3-bit repetition codes into a 9-qubit code that can detect both types of errors. The encoding process begins with the 3-bit encoding of the $\ket{+}$ state.

$$ \ket{+}_{\mathrm{3 bit}} = \frac{1}{\sqrt{2}}(\ket{000}+\ket{111})$$

Then, $\ket{0}_L$ is encoded by taking a tensor product of three $\ket{+}_{\mathrm{3 bit}}$ states.


$$ \ket{0}_L = \ket{+}_{\mathrm{3 bit}} \otimes \ket{+}_{\mathrm{3 bit}} \otimes \ket{+}_{\mathrm{3 bit}} $$
$$ \ket{0}_L = \frac{1}{\sqrt{8}}(\ket{000} + \ket{111})(\ket{000} + \ket{111})(\ket{000} + \ket{111})$$

The same process is completed for the $\ket{1}_L$ state, this time using $\ket{-}_{\mathrm{3 bit}}$ as the starting point.
$$ \ket{1}_L = \frac{1}{\sqrt{8}}(\ket{000} - \ket{111})(\ket{000} - \ket{111})(\ket{000} - \ket{111})$$

This encoding of $\psi = \alpha \ket{0} + \beta \ket{1}$ can be implemented with the following quantum circuit:

<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/shorencode.png?raw=1" alt="Drawing" style="width: 300px;"/>


The next consideration is to define the logical operators so that they behave as we expect, namely: $$\bar{X}\ket{0}_L = \ket{1}_L $$
$$\bar{X}\ket{1}_L = \ket{0}_L $$
and
$$\bar{Z}\ket{0}_L = \ket{0}_L$$
$$\bar{Z}\ket{1}_L = -\ket{1}_L.$$
In other words, we need the following equations to hold:

$$\bar{X}\ket{0}_L = \bar{X}\frac{1}{\sqrt{8}}(\ket{000} + \ket{111})(\ket{000} + \ket{111})(\ket{000} + \ket{111}) = \frac{1}{\sqrt{8}}(\ket{000} - \ket{111})(\ket{000} - \ket{111})(\ket{000} - \ket{111}) = \ket{1}_L  $$


$$\bar{Z}\ket{1}_L = \bar{Z}\frac{1}{\sqrt{8}}(\ket{000} - \ket{111})(\ket{000} - \ket{111})(\ket{000} - \ket{111}) = \frac{1}{\sqrt{8}}(\ket{111} - \ket{000})(\ket{111} - \ket{000})(\ket{111} - \ket{000}) = -\ket{1}_L$$


Can you see what the logical operators need to be?  


For a logical bit flip to occur ($\bar{X}$) the phase of each block needs to change.  This is accomplished by performing a $Z $ operation on one of the qubits in each block, thus $\bar{X} = Z_1Z_4Z_7$ is a valid choice, though not the only choice as others like $\bar{X} = Z_2Z_5Z_8$ or even $\bar{X} = Z_1Z_2Z_3Z_4Z_5Z_6Z_7Z_8Z_9$ also work. Similarly, for $\bar{Z}$ to take $\ket{1}_L$ to $-\ket{1}_L$ (and $\ket{0}_L$ to itself) all of the bits need to flip, thus  $\bar{Z} = X_1X_2X_3X_4X_5X_6X_7X_8X_9$. The curious reader can confirm that the anticommutativity holds between these logical operators and that they commute with each stabilizer discussed below.


Now, consider what happens when $X$ and $Z$-type errors corrupt a state encoded with the Shor code. If a bitflip error occurs on qubit 8 of the $\ket{0}_L$ state.

$$ X_8\ket{0}_L = \frac{1}{\sqrt{8}}(\ket{000} + \ket{111})(\ket{000} + \ket{111})(\ket{010} + \ket{101})$$

The error only corrupts the third block of the code which houses the 8th qubit. So, this means stabilizers that perform parity checks on that block only  ($Z_7Z_8$ and $Z_8Z_9$) are sufficient to determine which position experienced an error. Extending this, all bit flip errors can be corrected with the following six stabilizers, two for each block.  Because each block is an independent repetition code, the Shor code can handle three bit flip errors,  as long as they occur in distinct blocks.

$$ S_{\mathrm{bit flips}} = \{Z_1Z_2, Z_2Z_3, Z_4Z_5, Z_5Z_6, Z_7Z_8, Z_8Z_9\} $$

Now consider the impact of a a phase flip error that acts on the 6th qubit, for example.

$$ Z_6\ket{0}_L = \frac{1}{\sqrt{8}}(\ket{000} + \ket{111})(\ket{000} - \ket{111})(\ket{000} + \ket{111})$$

The phase of the second block is changed which means the entire state can be rewritten as $\ket{+}_{\mathrm{3 bit}} \otimes \ket{-}_{\mathrm{3 bit}} \otimes \ket{+}_{\mathrm{3 bit}}$.  This "zoomed out" view makes it clear how the repetition code is leveraged again. This time a stabilizer is needed which can test the parity of block 1 with block 2 and block 2 with block 3.

The stabilizer $X_1X_2X_3X_4X_5X_6$ can be used for this which will return 1 if the first two blocks have the same phase and -1 if they differ. Work this out by hand to convince yourself this works if it is not obvious why this is the case. Similarly, $X_4X_5X_6X_7X_8X_9$ can test the parity of the second two blocks completing the stabilizers necessary to detect phase flip errors.

$$ S_{\mathrm{phase flips}} = \{X_1X_2X_3X_4X_5X_6,X_4X_5X_6X_7X_8X_9   \} $$

All 8 stabilizers can correct any single-qubit $Z$ or $X$ error as summarized in the table below. Note that the Shor code is a redundant code, meaning that certain syndromes correspond to multiple errors.  At first this may seem problematic, but each error is fixed by the same correction, so knowing the specific source of the error is not always necessary.

| Error Type | Syndrome (Stabilizer Measurements) |
| ----------- | ----------- |
| No Error | 0 0 0 0 0 0 0 0 |
| $X_1$ | 1 0 0 0 0 0 0 0 |
| $X_2$ | 1 1 0 0 0 0 0 0 |
| $X_3$ | 0 1 0 0 0 0 0 0 |
| $X_4$ | 0 0 1 0 0 0 0 0 |
| $X_5$ | 0 0 1 1 0 0 0 0 |
| $X_6$ | 0 0 0 1 0 0 0 0 |
| $X_7$ | 0 0 0 0 1 0 0 0 |
| $X_8$ | 0 0 0 0 1 1 0 0 |
| $X_9$ | 0 0 0 0 0 1 0 0 |
| $Z_1$ | 0 0 0 0 0 0 1 0 |
| $Z_2$ | 0 0 0 0 0 0 1 0 |
| $Z_3$ | 0 0 0 0 0 0 1 0 |
| $Z_4$ | 0 0 0 0 0 0 1 1 |
| $Z_5$ | 0 0 0 0 0 0 1 1 |
| $Z_6$ | 0 0 0 0 0 0 1 1 |
| $Z_7$ | 0 0 0 0 0 0 0 1 |
| $Z_8$ | 0 0 0 0 0 0 0 1 |
| $Z_9$ | 0 0 0 0 0 0 0 1 |





## Exercise  3 - The Shor Code:

Now you have all of the backgound necessary to code the Shor code in CUDA-Q.  Fill in the sections below to build up a kernel that performs Shor code encoding and syndrome checks. The kernel should be constructed such that you can apply errors and select mesurement in the $Z$ or $X$ basis. Complete the tasks listed below to ensure your code works.

In [ ]:
import cudaq

@cudaq.kernel
def shor_code(error_qubit: list[int], error_location: list[int], measure: int):
    """Prepares a kernel for the Shor Code

    Parameters
    -----------
    error_qubit: list[int]
        a list where each element is an applied error designated as 1 =x or 2 =z
    error_location: list[int]
        each element corresponds to the index of the qubit which the error occurs on
    measure: int
        Option to measure in the z basis (0) or the x basis (1)

    Returns
    -------
    cudaq.kernel
        Kernel for running the Shor code
    """

    #Encode the data qubits with Shor encoding circuit.  Hint: It might be helpful to create separate registers for the data and ancilla qubits
    #TODO

    # Initial Psi (25/75) distribution in Z and X basis
    ry(1.04772,data_qubits[0])
    rz(1.521, data_qubits[0])

    # Apply optional single qubit errors
    #TODO

    # Apply Hadamard gate to ancilla qubits
    #TODO

    # Apply the Bit Flip syndromes
    #TODO

    # Apply the phase flip syndromes
    #TODO

    # Apply Hadamard gate to ancilla qubits
    #TODO

    # Perform mid-circuit measurements to determine syndromes
    #TODO


    # Apply the appropriate corrections based on the results from the syndrome measurements
    #TODO


    #Perform Hadamard on data qubits to rotate out of X basis (because of concatonated code)
    #TODO

    #Measure in X or Z basis depending on kernel input
    h(data_qubits) # put a Hadamard before the measurement to transform back into the Z basis
    # An X basis measurement can be obtained by applying a second Hadamard before a Z basis measurement
    #TODO

The final Hadamard is necessary because the code is concatenated and the second layer is in the $X$ basis.  Specifying the measurement basis allows you to confirm that the errors were or were not fixed.

You will also need to post process the results.  In the case of $Z$ basis measurement (where you see the impact of logical $X$ errors), you need to compute the parity of the logical $X$ operator $Z_1Z_2Z_3Z_4Z_5Z_6Z_7Z_8Z_9$, by measuring all the qubits in the $Z$ basis and computing the parity (Sum them and then mod 2) of the results.  

The same can be done for an $X$ basis measurement (where you see the impact of logical $Z$ errors). In this case you need to compute the parity of the logical $Z$ operator $X_1X_2X_3X_4X_5X_6X_7X_8X_9$ by measuring all of the qubits in the $X$ basis and computing thier parity.

Write a postprocessing function below which takes results, computes the parity of each measurment, prints the number of 1's and 0's, and prints the results.

In [ ]:
def post_process(results):
    """takes results from a CUDA-Q sample and prints the results and the number of 0's and 1's by computing the parity of the bitstrings.

    Parameters
    -----------
    results: cudaq.SampleResult
                A dictionary of the results from sampling the quantum state
    """


Now, run your code through the following tests and confirm it is working well.

1. Prepare $\ket{\psi}$ in the $\ket{0}$ state.  Sample your kernel with no errors in the $Z$ and $X$ basis.  Do you see a 100/0 and 50/50 distribution for each respectively? What do you notice about the bitstrings when you measure in each basis?

2. Now, comment out the part of your code that fixes errors. Add a single $Z$ error and measure in the $Z$ and $X$ basis. Do you observe a bitflip in the $Z$ basis results?  Note, because the Shor code has an extra layer of Hadamard gates, a $Z$ error impacts the $Z$ observable which is not the usual case.

3. Now prepare $\ket{\psi}$ in the $\ket{+}$ state. Comment out the part of your code that fixes errors, add a single $X$ error, and measure in the $Z$ and $X$ basis. Do you observe a bitflip in the $X$ basis results now?

4. Uncomment the part of your code that fixes the errors and run the same samples in 2 and 3. Did the correct syndrome flag and was the impact of the error ameliorated?

5. Prepare $\ket{\psi}$ in the $\ket{0}$ state again. With your full code, add multiple $Z$ errors and note that the stabilizer checks cannot properly fix them as the code is only distance 3.

In [ ]:
# TODO
# Run tests on your Shor Code

# Decoders

$\renewcommand{\ket}[1]{|#1\rangle}$

In [ ]:
!pip install cudaq -q
!pip install torch scikit-learn galois ipywidgets -q


> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).

In [ ]:
import sys
import os
from itertools import product

import numpy as np
import matplotlib.pyplot as plt
import requests
import bz2
import galois
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

import cudaq
from cudaq import spin
from cudaq.qis import *

## To install cudaq-qec (if not already installed), uncomment and run:
## !pip install cudaq-qec -q
import cudaq_qec as qec

from Images.decoder.decoder_widget import display_widget
from Images.decoder.bp import run_decoder, parse_csr_mat, parse_H_csr, parse_obs_csr

cudaq.set_target('nvidia')

## Decoding Decoded

Remember that a QEC round involves four main steps:
* Encoding logical qubits and sending the system through a noisy channel, which is often just a duration of time where the qubits are exposed to potential sources of error
* Measuring syndrome data
* Decoding the syndrome to identify where an error occurred and what instructions to send to the QPU to fix the error
* Correcting the error

<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/decoding.png?raw=1" alt="Diagram showing the four main steps of a QEC round: encoding, syndrome measurement, decoding, and error correction, with the decoder receiving syndrome data and sending corrections to the QPU" style="width: 900px;"/>

The decoding step is very challenging and is considered one of the primary limitations for QEC. This is because decoding requires measurements on a QPU, data transfer to the supercomputer, decoding on the supercomputer, and then data transfer back to the QPU.  The time available for this is called the **decoding window** and varies based on a number of factors such as the qubit modality, data transfer rates, and the volume of information that needs to be decoded.

Directly competing with speed is accuracy. If a decoder is inaccurate, errors will be missed or introduced each QEC round and will propagate to ruin the computation. High-distance codes are necessary for accuracy, but unfortunately introduce high-qubit overheads and make decoding much more challenging. Advances in QEC code design and low-latency integration between AI supercomputers and QPUs alleviate pressure on the decoding step, but it nevertheless remains the primary bottleneck of QEC.




## Pauli Frames and Error Tracking

In practice, when errors are identified by the decoder, they are not immediately corrected but are tracked using a **Pauli frame**. The Pauli frame keeps track of the corrections classically and applies them later. This approach reduces the number of gate operations required to fix errors on the QPU, thereby protecting the encoded state from additional noise introduced by each correction gate. For instance, if a bit flip error occurs on qubit 1 in the first round and another bit flip error happens on the same qubit later, the two errors cancel each other out, eliminating the need for a correction

Often, codes are depicted using 3D images like the one below. In this case, each plane is a Steane code QEC round with flagged errors in purple. Each error is saved, and the list grows with future rounds. The final Paul frame, $[X_1, X_5, X_1]$, is the list of corrections for the three bit flip errors that have occurred over all the rounds: two on qubit 1 and one on qubit 5.  In the last step, the errors can be simplified, for example, $X_1X_1 = I$, so only one of the three corrections, $X_5$, needs to be applied. This is a rather trivial example, and often diagrams like this are used to depict more complex codes and their respective error pathways.

<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/pauliframes.png?raw=1" alt="3D visualization of Pauli frame tracking across multiple Steane code QEC rounds, showing purple error markers on successive planes and the accumulated Pauli frame corrections" style="width: 700px;"/>

The dimension of time can also lend itself to more sophisticated decoding schemes. This is particularly important when measurement errors occur during the stabilizer checks. In this case, it might appear that a stabilizer flags when in fact the data qubits are fine. Multiple decoding rounds can demonstrate that the false stabilizer flag is a consequence of measurement error and not a true error, where other true errors would persist without correction. Such an approach is more powerful but requires decoding of much more complex syndromes. The diagram below demonstrates this concept with an example.

<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/decodeintime.png?raw=1" alt="Diagram comparing single-round decoding versus decoding over multiple time steps, showing how measurement errors can be detected when syndrome data from consecutive rounds is analyzed together" style="width: 1000px;"/>
Notice how, in the first case, the decoder has kept track of a measurement error and is therefore making an incorrect syndrome in the final case. When decoding happens over time, the decoding task must not decode a 19-bit syndrome but is able to flag measurement errors



### Exercise 1:

The benefit of decoding in time is that the measurement errors can be factored into the decoding process.  However, the tradeoff is that the decoding problem is much harder.  When decoding in time, an effective parity check matrix must be constructed as an input to the decoder. In this exercise you will build $H^{(2)}$ for a two round Steane code that includes consistency checks to  flag errors between the two time steps.  

First, a few hints.  Consider the dimensions. The number of columns still corresponds to the number of qubits, but, now we need to take into account the data qubits at time 0, the data qubits at time 1, and the three ancilla qubits used to measure syndromes between the two rounds.

Each time step will have the same three stabilizer checks, so $H^{(2)}$ should have six rows.

The top left block of $H^{(2)}$ and the bottom right block will be $H$, where $H$ is the standard Steane code parity check matrix.

What do the middle three columns need to be for $H^{(2)}$ to be able to catch measurement errors?

Build $H^{(2)}$, and then build an error vector $e$ of size 17 such that each entry is a 0 or a 1 if an error occurred on that qubit.  Compute $H^{(2)}e^T$ for a case with an error on data qubit 1 in the first time step, an error on data qubit 1 in the second time step only, and a measurement error. Note, it is best practice to assume that the decoder will not hanndle raw syndrome outputs, but the differences between he current set of measurements and the next round.  For example, after preparation the syndrome 101 might be measured. If the next round produces the same stabilizer measurerments, the decoder would see 000 not 101. This syntax makes it much easier for decoders to handle data in more complex settings.



In [ ]:
# EXERCISE 1
H = np.array([
    [1, 1, 0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0, 1, 0],
    [0, 1, 1, 1, 0, 0, 1]
])

# Build 2 round parity check matrix.
H2 = np.array([##TODO##
])

# Syndrome for no error
e = np.array([##TODO##])

print(H2 @ e.T)

# Syndrome for error on first data qubit in first time step
e = np.array([##TODO##])

print(H2 @ e.T)

# Syndrome for error on first measurement qubit
e = np.array([##TODO##])

print(H2 @ e.T)

# Syndrome for error on first data qubit in second time step
e = np.array([##TODO##])

print(H2 @ e.T)

# Try other errors

Looking at your results. Can you see how a measurement error can be detected in the the symdrome pattern? Note that the parity check matrix you created can only catch measurment errors on round 1.  A round 2 measurement error would be missed unless you extenced the parity check matrix further.  

## Most Likely Error Decoding

So far, decoders have been presented as black boxes. In many cases that is sufficient.  If you are developing or testing new codes, you might just use a state of the art decoder and not care how it works.  In other cases, the opposite is true, and you might focus on developing and tuning a decoder to work well for a specific sort of QEC situation.

The rest of this lab will allow you to explore a number of different decoders ranging from conceptually simple to state of the art.  You will interact with them in different ways, in certain cases you'll write the decoder from scratch and at other times you'll use the decoder out of the box.

The starting point is to consider a naive brute forced decoder that is conceptually simple yet sets the upper bound on decoder accuracy.  

The steps of **maximum likelihood decoding** are as follows (considering only bitflip errors for simplicity):

1. Select a QEC code and encode a message in the codespace with $n$ data qubits.
2. Generate the $2^n$ bitstrings $\{x_0, \cdots, x_{2^n} \}$ of length $n$ corresponding to all possible error situations.
3. For each $x_i$, compute the syndrome as $Hx_i~\mathrm{mod} 2$ where $H$ is the code's parity check matrix.
4. Then, for each possible syndrome, list errors that could have produced the given syndrome.
5. Under each syndrome, order errors by their Hamming distance from the original message.
6. Finally, the decoder will receive a new syndrome, and then select the lowest Hamming distance error and apply the fix.

The assumption here is that, if errors are independent, cases with fewer errors are always more likely than those with more errors.

Notice, in previous lab when you coded the Steane code, we assumed a situation where one error occurs at a time, allowing your syndrome checks to fix errors.  This is the same assumption made here. The problem with this approach is that it does not scale.  There are $2^n$ errors that need to be computed *a priori* which is not possible for large codes.  The sections below will consider more scaleable heuristics to overcome this issue.





### Exercise 2:

Code the most likely error decoder for the Steane code below given the parity check matrix below.  For each syndrome, print the associated length 7 bitstrings that produce that error, the Hamming distance from the baseline message (0000000), and the probability of that error.



In [ ]:
# EXERCISE 2
# Define the bit-flip probability
p = 0.1

# Define the parity check matrix H
H = np.array([
    [1, 1, 0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0, 1, 0],
    [0, 1, 1, 1, 0, 0, 1]
])

# Generate the syndromes and sort based on the instructions above
##TODO##

## AI Decoders

One way to circumvent the scaling challenges posed by the a brute force most likely error decoder is to use tools like AI. AI is fantastic at pattern recognition, runs very quickly, and can easily scale.

AI decoders also offer flexibility as they can be trained with simulated data or even trained on small-distance codes and, via transfer learning, be used to decode higher distance codes.

Recently, [NVIDIA and QuEra announced a new transformed based decoder](https://developer.nvidia.com/blog/nvidia-and-quera-decode-quantum-errors-with-ai/) tested on magic state distillation circuits used by QuEra (A 35 qubit circuit with 5 Steane code logically encoded logical qubits).  The decoder showed promise by outperforming the decoder used by QuEra in terms of speed and accuracy.  Additionally, the AI decoder might have the potential to scale to code distances large enough for sufficiently low logical error rates.

<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/aidecoderplot.png?raw=1" alt="Performance comparison plot of the NVIDIA-QuEra AI transformer decoder versus baseline, showing improved accuracy and speed on magic state distillation circuits" style="width: 700px;"/>

### Exercise 3:

You will now build a working AI decoder for the Steane code. The goal is to build something similar to the workflow in the image below.

<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/aidecoderworkflow.png?raw=1" alt="Workflow diagram showing the AI decoder pipeline: CUDA-Q generates noisy syndrome data, which is split into training and test sets to train a neural network that predicts logical errors" style="width: 700px;"/>

This lab does not expect you to have experience coding AI models with tools like PyTorch, so you will focus on the data generation and learn how to prepare the data to train an AI decoder without worrying about details of the model. Follow the steps outlined below to complete the code.



The first step is to generate the training data. Take the Steane code circuit you coded in Lab 2, now with bitflip noise to each qubit after encoding.  In this case, we can explore circuit-level noise based on simulated results rather than a contrived data set.

Create a data set of 5000 samples.  To generate this, run `cudaq.run()` 5000 times taking one shot each time.  Output the measurements from the syndrome checks (without correcting any errors) and then measure all of the data qubits.  Compute the parity of bits corresponding to the correct logical operator to determine the true logical state.  

Save the syndromes and the logical states as two numpy arrays.  This will be your data set.

In [ ]:
# EXERCISE 3
p = 0.05
cudaq.unset_noise()
noise = cudaq.NoiseModel()

@cudaq.kernel
def steane_code() -> list[int]:
    """Prepares a kernel for the Steane Code
    Returns
    -------
    cudaq.kernel
        Kernel for running the Steane code
    """
    data_qubits = cudaq.qvector(7)
    ancilla_qubits = cudaq.qvector(3)

    # Create a superposition over all possible combinations of parity check bits
    h(data_qubits[4])
    h(data_qubits[5])
    h(data_qubits[6])

    #Entangle states to enforce constraints of parity check matrix

    x.ctrl(data_qubits[0],data_qubits[1])
    x.ctrl(data_qubits[0],data_qubits[2])

    x.ctrl(data_qubits[4],data_qubits[0])
    x.ctrl(data_qubits[4],data_qubits[1])
    x.ctrl(data_qubits[4],data_qubits[3])

    x.ctrl(data_qubits[5],data_qubits[0])
    x.ctrl(data_qubits[5],data_qubits[2])
    x.ctrl(data_qubits[5],data_qubits[3])

    x.ctrl(data_qubits[6],data_qubits[1])
    x.ctrl(data_qubits[6],data_qubits[2])
    x.ctrl(data_qubits[6],data_qubits[3])

    for j in range(7):
        cudaq.apply_noise(cudaq.XError, p, data_qubits[j])

    # Detect X errors
    h(ancilla_qubits)

    z.ctrl(ancilla_qubits[0],data_qubits[0])
    z.ctrl(ancilla_qubits[0],data_qubits[1])
    z.ctrl(ancilla_qubits[0],data_qubits[3])
    z.ctrl(ancilla_qubits[0],data_qubits[4])

    z.ctrl(ancilla_qubits[1],data_qubits[0])
    z.ctrl(ancilla_qubits[1],data_qubits[2])
    z.ctrl(ancilla_qubits[1],data_qubits[3])
    z.ctrl(ancilla_qubits[1],data_qubits[5])

    z.ctrl(ancilla_qubits[2],data_qubits[1])
    z.ctrl(ancilla_qubits[2],data_qubits[2])
    z.ctrl(ancilla_qubits[2],data_qubits[3])
    z.ctrl(ancilla_qubits[2],data_qubits[6])

    h(ancilla_qubits)

    d0=mz(data_qubits[0])
    d1=mz(data_qubits[1])
    d2=mz(data_qubits[2])
    d3=mz(data_qubits[3])
    d4=mz(data_qubits[4])
    d5=mz(data_qubits[5])
    d6=mz(data_qubits[6])
    a0=mz(ancilla_qubits[0])
    a1=mz(ancilla_qubits[1])
    a2=mz(ancilla_qubits[2])

    return [d0,d1,d2,d3,d4,d5,d6,a0,a1,a2]

# Generate Data
##TODO## - generate the inout data

raw_logical_error_rate =##TODO## - calculate the raw logical error rate

The previous cell is quite a bit of work, and requires you to manually construct the entire QEC code. This is not ideal when you are primarily interested in testing an AI decoder and want data generation streamlined.

Another more efficient way to sample training data for memory experiments is directly using the parity check matrix. Random bitflips can be applied and syndromes determined via matrix multiplication. CUDA-Q QEC can do this with just a few lines of codes (shown below) and generate the same sort of data we did above with a preloaded Steane code. Additionally, if you want to generate data for multiple syndrome extraction rounds, you can use the sample_memory_circuit. If you want to test a new, non-standard code, you would need to define the kernels explicitly similar to the example above as shown in the docs [here](https://nvidia.github.io/cudaqx/components/qec/introduction.html#qec-code-framework-cudaq-qec-code)

In [ ]:
steane = qec.get_code("steane")

Hz = steane.get_parity_z()

observable = steane.get_observables_z()

data = qec.generate_random_bit_flips(Hz.shape[1], p)

syndrome = Hz @ data % 2
print(f"syndrome: {syndrome}")

actual_observable = observable @ data % 2
print(f"actual_observable: {actual_observable}")

Your data set will be split into a 4000 point training set and a 1000 point test set to validate that the model worked.  The inputs to the model will be syndrome data obtained from your simulations.  The loss function will be a comparison between the predicted logical state of the model and the logical state you obtained from measuring the data qubits.

If you defined the arrays properly, the code below should preprocesses the data for you, splitting it into a training and test set, construct a simple neural network, define model setting, and train the model.

Each epoch is one trining loop through the entire data set.  The model will then test the accuracy using the test set which was excluded from training at each epoch. The code is provided for the interested reader to see the details of the torch implementation, can also just be run as a black box. Do the plots below indicate the model improved as training progressed?

In [ ]:

# This section normalizes and loads the data you defined previously
syndromes = np.array(syndromes, dtype=np.float32)
logical_flips = np.array(logical_flips, dtype=np.float32)

# Normalize input data
syndromes = (syndromes - syndromes.mean()) / syndromes.std()

X_train, X_test, y_train, y_test = train_test_split(
    syndromes, logical_flips, test_size=0.20, random_state=42)

X_train = torch.tensor(X_train)
X_test = torch.tensor(X_test)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

# Create data loaders
batch_size = 32
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=True
)

# This builds a simple NN model which will be trained
class SimpleDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)
        self.act = nn.Sigmoid()

        # Initialize weights
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.xavier_uniform_(self.fc3.weight)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.act(self.fc3(x))

model = SimpleDecoder()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=10,)

# Define the test_accuracy function
def test_accuracy():
    model.eval()
    with torch.no_grad():
        pred = (model(X_test).squeeze() > 0.5).float()
        return (pred == y_test).float().mean().item()

# Training loop
num_epochs = 30
train_losses, test_acc = [], []

# epoch 0 (untrained)
test_acc.append(test_accuracy())

for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_loss = 0
    batch_count = 0

    # Training
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        out = model(batch_X).squeeze()
        loss = criterion(out, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        batch_count += 1

    # Calculate average loss for the epoch
    avg_epoch_loss = epoch_loss / batch_count
    train_losses.append(avg_epoch_loss)

    # Evaluate accuracy
    current_acc = test_accuracy()
    test_acc.append(current_acc)

    # Update learning rate
    scheduler.step(avg_epoch_loss)

    if epoch % 1 == 0:
        print(f"Epoch {epoch:3d}/{num_epochs} | "
              f"train loss={avg_epoch_loss:.4f} | "
              f"test acc={current_acc:.4f}")

# Plotting
plt.figure(figsize=(12,4))

plt.subplot(1, 2, 1)
plt.plot(range(num_epochs + 1), test_acc, marker='o')
plt.xlabel("Training Epoch")
plt.ylabel("Test Accuracy")
plt.title("Steane-Code Decoder Accuracy")
plt.grid(True)
plt.ylim(0, 1)

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), train_losses, color='red')
plt.xlabel("Training Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Over Time")
plt.grid(True)

plt.tight_layout()
plt.show()

# Print final metrics
print(f"\nFinal Results:")
print(f"Final Test Accuracy: {test_acc[-1]:.4f}")
print(f"Final Training Loss: {train_losses[-1]:.4f}")
print(f"Raw (Undecoded) Accuracy: {1-raw_logical_error_rate:.4f}")

# Print first 20 predictions
model.eval()
with torch.no_grad():
    test_predictions = model(X_test).squeeze()
    predicted_labels = (test_predictions > 0.5).float()

print("\nFirst 20 test examples:")
print("=" * 40)
print(f"{'Index':^6} | {'True':^6} | {'Predicted':^6}")
print("=" * 40)

for i in range(20):
    true_label = int(y_test[i].item())
    pred_label = int(predicted_labels[i].item())
    print(f"{i:^6} | {true_label:^6} | {pred_label:^6}")

print("=" * 40)

# Calculate accuracy for these 20 examples
correct = (predicted_labels[:20] == y_test[:20]).sum().item()
print(f"\nAccuracy for these 20 examples: {correct}/20 = {correct/20:.2%}")



You should see the model successfully train! The test set accuracy should increase while the loss functions decreases.

There are a few key observations to discuss.

1. If the model parameters are random and we run this training multiple times, we should see the model on average start with an accuracy of about 0.5. This means we would have as much luck flipping a coin as our decoder. It may start higher or lower depending on the initial parameters bias towards outputting 1's or 0's. So, we do demonstrate the model did train.

2. The trained model does outperform the raw logical error rate without decoding. So, our AI decoder is an improvement. Given the simplicity of the Steane code, this is unsurprising, as there is not really any hidden insight to be gleaned as we can essentially work out the brute force MLE decoding by hand.

3. The final output of the AI model (or any decoder) is limited by the underlying QEC code and its distance. This is what determines what errors are detectable or correctable before we even try decoding. For example, the distance three Steane code cannot correct two errors. So, we cannot expect the AI model to learn how to correct these errors either. Thus, AI decoding shines in the regime where there are many correctable errors with non-trivial syndrome patterns.

4. Because the trained decoder depends on the error model used, it is really important to have large training data sets with sufficiently realistic noise models and model often require fine tuning with experimental data to realize peak performance. If the error rate was tiny, the model may just learn to output logical 0 all the time, and learn nothing about the error patterns if it has insufficient cases to train on. When training on physical QPU data, we are not trying to learn a noise model, but the actual, unknown noise profile of the device.

## Belief Propagation Decoding

Another state-of-the-art decoding method is **belief propagation (BP)**.  BP is a powerful technique borrowed from classical error correction that is highly flexible and can serve as a black box decoder for arbitrary QEC Codes. It is particularly useful for codes like quantum low-density parity check (**qLDPC**). All the user needs to do is provide a parity check matrix and then feed the decoder syndromes to decode.

NVIDIA created a GPU accelerated BP decoder which allows researchers to push QEC even further than before.  This section will walk you through implementing BP and how to use NVIDIA's accelerated BP decoder.

At a high level, BP works by taking initial beliefs about each data qubit's physical error rate and, using syndrome measurements, iteratively update these beliefs to converge on a solution which provides a likelihood of an error occurring or not.  

Though there are different variations of BP, one implementation is the following:
1. Initialize a log-likelihood ratio $L_{v_i} = log(\frac{1-p}{p})$ for each data qubit $v_i$ where $p$ is the physical error rate.  If an error is unlikely, the quantity is positive and increases in magnitude the smaller $p$ is as errors are less likley.
   
2. $L_{v_i\rightarrow c_j}$, the message sent from each data qubit to a check qubit, is calculated based on the following equation:  $L_{v\rightarrow c} = L_i + \sum_{k \in N(i)/j} L_{c_k\rightarrow v_i}$ where $\sum_{k \in N(i)/j}$ includes all of the check qubits connected to data qubit $v_i$ excluding check qubit $c_j$. That is, the check qubit is updated with beliefs the variable learned from the *other* check qubits it is connected to.

   
3. Similarly, $L_{c_j\rightarrow v_i}$ is computed to compute the messages sent back to the variable qubits using $L_{c_j \to v_i} = (-1)^{s_j} \cdot 2 \cdot \text{arctanh} \left( \prod_{k \in N(j) \setminus i} \tanh \left( \frac{L_{v_k \to c_j}}{2} \right) \right)$ where $N(j) \setminus i$ is all *other* variable nodes connected to $c_j$ excluding $v_i$ and $s_j$ is the value of the syndrome measurement for $c_j$.
   
4. Finally, the final beliefs $L_{\text{final}, i}$ are computed as $L_i + \sum_{j \in N(i)} L_{c_j \to v_i}$, summing the prior beliefs with the final messages sent to each variable node. From this a decision can be made where positive numbers indicate no error and negative an error, with the magnitudes related to confidence.

Ideally, BP will converge to a solution that agrees with the original syndrome and correct the error.  If BP cannot converge, it means there is still significant uncertainty whether some of the bits have errors or not and postprocessing is necessary to refine the result.  This will be discussed in the following section.


### Exercise 4:

Below is the start of a BP implementation for decoding the 5-qubit repetition code. Fill in the sections marked "TODO" to complete the code.  Most of the BP loops are calculated for you.  Make sure to review them and understand what is going on.  Then, you will complete the code by fixing the code to calculate the final belief on each qubit and determine where errors occurred.


In [ ]:
# EXERCISE 4
physical_error_rate = 0.1
max_iter = 5

H = np.array([
    [1, 1, 0, 0, 0],
    [0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0],
    [0, 0, 0, 1, 1]
], dtype=int)

n_qubits = H.shape[1]
n_checks = H.shape[0]

actual_error = np.array([0, 0, 1, 0, 0])

syndrome = H @ actual_error % 2

print("--- Simulation Setup ---")
print(f"Physical Error Rate (p): {physical_error_rate}")
print(f"Actual Error Vector:     {actual_error}")
print(f"Resulting Syndrome:      {syndrome}")
print("-" * 40 + "\n")


#Initialize the prior probabilites, and matricies for v -> c and c -> v messages
L_prior = np.log((1 - physical_error_rate) / physical_error_rate)

L_v_to_c = np.zeros((n_qubits, n_checks))

L_c_to_v = np.zeros((n_checks, n_qubits))


print("--- Initialization ---")
print(f"Prior LLR (L_i) for all qubits: {L_prior:.4f}\n")



for iteration in range(max_iter):
    print(f"--- Iteration {iteration + 1} ---")


    # Compute L_{v_i -> c_j} = L_i + sum_{k in N(i)\j} L_{c_k -> v_i}
    # The message is the qubit's own belief (prior) plus all incoming messages from checks

    for i in range(n_qubits):
        for j in range(n_checks):
            if H[j, i] == 1:
                connected_checks = np.where(H[:, i] == 1)[0]
                sum_incoming_L = 0
                for k in connected_checks:
                    if k != j:
                        sum_incoming_L += L_c_to_v[k, i]
                L_v_to_c[i, j] = L_prior + sum_incoming_L

    print("Step A: Updated Variable-to-Check Message Matrix (L_v_to_c):")
    print(np.round(L_v_to_c, 4))
    print("")


    # Compute L_{c_j -> v_i} = (-1)^s_j * 2 * atanh( product_{k in N(j)\i} tanh(L_{v_k -> c_j}/2) )
    # The message is based on the check's syndrome value and all of the incoming messages from the data qubits.

    for j in range(n_checks):
        for i in range(n_qubits):
            if H[j, i] == 1:
                connected_qubits = np.where(H[j, :] == 1)[0]
                prod_tanh = 1.0
                for k in connected_qubits:
                    if k != i:
                        # tanh can be unstable for large LLRs, so we clip its argument
                        val = np.clip(L_v_to_c[k, j] / 2, -20, 20)
                        prod_tanh *= np.tanh(val)

                sign = (-1)**syndrome[j]

                # FIX IS HERE: Clip prod_tanh to avoid arctanh(1) or arctanh(-1)
                # This defines the missing variable 'prod_tanh_clipped'
                epsilon = 1e-12
                prod_tanh_clipped = np.clip(prod_tanh, -1 + epsilon, 1 - epsilon)

                L_c_to_v[j, i] = sign * 2 * np.arctanh(prod_tanh_clipped)



    print("Step B: Updated Check-to-Variable Message Matrix (L_c_to_v):")
    print(np.round(L_c_to_v, 4))
    print("")


##TODO## Compute the final belief of each qubit (prior beliefs plus sum of incoming messages)


##TODO## END

print("--- Final Belief ---")
print(f"Final LLRs: {np.round(L_final, 4)}")


estimated_error = ##TODO## Determine where errors occurred based on L_final

print("-" * 40)
print(f"Decoder Estimated Error:  {estimated_error}")

if np.array_equal(actual_error, estimated_error):
    print("\n✅ Success: The decoder correctly identified the error.")
else:
    print("\n❌ Failure: The decoder did not find the correct error.")

The 5-qubit repetition code is small, and easy to decode.  If more than one error is present, the BPO decoder will fail, as the code distance is the primary constraint. However, with more complex codes, BP may complete, but not converge, meaning uncertainty remains on some of the bits and a logical error may occur due to its failure. Also, decoding large codes with orders of magnitude more variable and check nodes can become cumbersome.

NVIDIA's accelerated BP decoder, now part of the CUDA-Q QEC library, helps address this problem and parallelizes the BP algorithm.  Examining your code above, it is not hard to see how the message calculations can be trivially parallelized, making them well suited for GPU acceleration.

Try running NVIDIA's accelerated BP decoder by running the code below which imports a data set of syndrome data from a large qLDPC code.  How long did it take to decode 10 shots?  


In [ ]:
if __name__ == "__main__":
    # See other test data options in https://github.com/NVIDIA/cudaqx/releases/tag/0.2.0
    filename = 'osd_1008_8785_0.001.json' # lower error rate
    bz2filename = filename + '.bz2'
    if not os.path.exists(filename):
        url = f"https://github.com/NVIDIA/cudaqx/releases/download/0.2.0/{bz2filename}"
        # Download the file
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Raise an error if download fails
        with open(bz2filename, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        print(f'Decompressing {bz2filename} into {filename}')

        # Decompress the file
        with bz2.BZ2File(bz2filename, "rb") as f_in, open(filename,
                                                          "wb") as f_out:
            f_out.write(f_in.read())

        print(f"Decompressed file saved as {filename}")

In [ ]:
if __name__ == "__main__":
    num_shots = 10
    run_as_batched = True
    print_output = True
    osd_method = 0 # 0 is off 1 is on
    run_decoder(filename, num_shots, run_as_batched, print_output, osd_method)

One of the reasons this algorithm can be accelerated is that it can be naturally parallelized. Do you see where that might happen?  

The answer lies in the process of sending messages from syndrome qubits to data qubits, and vice versa. Each of these messages can be computed in parallel. This corresponds to performing operations that alternate across the rows and columns of a matrix, depending on which message is being sent. Each row and column can be treated independently, allowing for further parallelization within each row or column.

In benchmarks of large code instances, the NVIDIA decoder was up to 35x faster than the industry standard implementation for benchmarks run on the [[144,12,12]](https://arxiv.org/abs/2308.07915) code.

<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/benchmark.png?raw=1" alt="Bar chart showing NVIDIA GPU-accelerated BP decoder performance benchmarks, demonstrating up to 35x speedup over industry standard implementations on the [[144,12,12]] qLDPC code" style="width: 700px;"/>



### Batching

The performance can be enhanced ever further using batching.  Batching means that a collection of syndromes are sent to the GPU and decoded simultaneously.  This minimizes data transfer and allows the entire GPU to be working, reducing the average decoding time per shot.  

Run the cells below to see the impact of batching when decoding 10000 syndromes. In the second cell, `run_as_batched` is set to `True`. How much faster is batching?  Batching is most effective when decoding a large number of syndromes.

In [ ]:
if __name__ == "__main__":
    num_shots = 10000
    run_as_batched = False
    print_output = False
    osd_method = 0 # 0 is off 1 is on
    run_decoder(filename, num_shots, run_as_batched, print_output, osd_method)

In [ ]:
if __name__ == "__main__":
    num_shots = 10000
    run_as_batched = True
    print_output = False
    osd_method = 0 # 0 is off 1 is on
    run_decoder(filename, num_shots, run_as_batched, print_output, osd_method)

### Ordered Statistics Decoding
Now, increase the error rate using a different data file from 0.1 % to 0.5 %.  What happens to the logical error rate?

In [ ]:
if __name__ == "__main__":
    # See other test data options in https://github.com/NVIDIA/cudaqx/releases/tag/0.2.0
    filename = 'osd_1008_8785_0.005.json' # lower error rate
    bz2filename = filename + '.bz2'
    if not os.path.exists(filename):
        url = f"https://github.com/NVIDIA/cudaqx/releases/download/0.2.0/{bz2filename}"
        # Download the file
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Raise an error if download fails
        with open(bz2filename, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        print(f'Decompressing {bz2filename} into {filename}')

        # Decompress the file
        with bz2.BZ2File(bz2filename, "rb") as f_in, open(filename,
                                                          "wb") as f_out:
            f_out.write(f_in.read())

        print(f"Decompressed file saved as {filename}")

In [ ]:
if __name__ == "__main__":
    num_shots = 10000
    run_as_batched = True
    print_output = False
    osd_method = 0 # 0 is off 1 is on
    run_decoder(filename, num_shots, run_as_batched, print_output, osd_method)

You should notice a very high logical error rate. Each of these logical error rates is a case where BP could not converge.  Like any numerical solver, it is possible to increase iterations and perform a number of tricks to help BP, but some particularly difficult syndromes will not converge either way. To solve this, BP decoders are often paired with optional ordered statistics decoding (OSD) which performs post-processing to improve the result.  Thus, you might often see a "BP+OSD" decoder.

The intuition behind OSD decoding is to solve for the errors directly using matrix inversion.  That is, solving $e = H^{-1}s$ where $e$ is a vector with error locations, $s$ is the syndrome, and $H$ is the parity check matrix.  The procedure cannot be performed with the original $H$, so instead, a square matrix with columns of full-rank must be constructed from the columns of $H$.

What makes this approach challenging is selecting the right columns.  Many different solutions can be obtained with different column selections, but without guidance, this will likely result in a solution that is not of minimum weight and induce a logical error. Recall, for the most likely error decoder above, that valid errors with high weight are much less likely than low weight errors.  Same idea here, but rather than computing all combinations, the OSD heuristic provides a clever way to search from a really good starting point that is the output of BP.

The output of BP assigns a probability of error to each data qubit.  This naturally informs the reordering of the columns to rank from most to least likely to have an error. Next Gaussian elimination is performed on $H$ to determine the first full columns of full rank. These subcolumns are then inverted to get $H^{-1}$ which is multiplied by $s$ to get the error result $e_{[s]}$. The $e_{[s]}$ is then padded with zeros  $e_{[T]} =0$ which results in a total error profile of $ e = (e_{[s]},e_{[T]})$.  


### Exercise 5:

The following exercise is based on a [lecture](https://www.youtube.com/watch?v=b9N2Ps3FTto) by Joschka Roffe. Given the parity check matrix below and the probabilities of error from BP. Perform OSD manually and find the error profile that satisfies the syndrome. Note, all computations must be performed using mod 2 arithmetic. This can be accomplished using the `galois` library which creates a Galois field and allows all `numpy` operations to compute mod 2.


In [ ]:
# EXERCISE 5
GF2 = galois.GF(2) # allows mod 2 math.

#parity check matrix as numpy array
H = np.array([[0, 1, 0, 0, 1, 0],
              [0, 0, 1, 0, 0, 1],
              [1, 0, 0, 1, 0, 0],
              [1, 0, 0, 0, 1, 1]])

bp_results = [0.05, 0.5, 0.01, 0.01, 0.82, 0.05]

s = GF2([1,0,0,1]) #syndrome

# Get indices that would sort bp_results in descending order
##TODO##

# Rearrange columns of numpy array
##TODO##
H_sorted =


# Convert to GF2 matrix after rearrangement for mod 2 aritmatic
GF2 = galois.GF(2)
H_sorted = GF2(H_sorted)

# Perform Gaussian elimination to identify the first four columns of full rank.
def rref(matrix):
    rows, cols = matrix.shape
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        if matrix[r, c] == 0:
            for i in range(r + 1, rows):
                if matrix[i, c] != 0:
                    matrix[[r, i]] = matrix[[i, r]]
                    break
        if matrix[r, c] == 0:
            continue
        matrix[r] = matrix[r] / matrix[r, c]
        for i in range(rows):
            if i != r and matrix[i, c] != 0:
                matrix[i] = matrix[i] - matrix[i, c] * matrix[r]
        r += 1
    return matrix

print("Gaussian Elimination Result")
print(rref(H_sorted.copy())) # First four columns are pivot columns

# Build H_s from the first full rank columns
##TODO##

# Compute Hs_inverse
##TODO##

# Calculate e_s
##TODO##

# Pad result with zeros and reorder based on colum sorting from earlier.
##TODO##

# Confirm that the errors produce the expected syndrome from the original H
##TODO##

from Images.decoder.solution3_button import show_cudaq_solution
show_cudaq_solution()

OSD guarantees an error pattern that satisfies the syndrome, but it is not necessarily a minimum weight solution nor avoids a logical error. In the exercise above, you performed OSD-0, or zero order OSD.  To improve results, higher order OSD can be performed.  Higher order OSD performs bitflips in the $e_{[T]}$ bits and checks if a lower weight solution can be found. Generally speaking, the order of the OSD determines how many bitflips are considered, but there is additional nuance not covered here.

Lower weight solutions are possible as bit flips in $e_{[T]}$ also impact the other bits in the $e_{[s]}$ subspace.

To summarize, syndromes are of varying difficulty. Easy syndromes are solved with BP, OSD-0 is used for moderate syndrome difficulties, and higher order OSD is used for the most challenging.  


<img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/bposd.png?raw=1" alt="Flowchart illustrating the BP+OSD decoding pipeline: BP attempts to converge on easy syndromes, OSD-0 handles moderate cases, and higher-order OSD processes the most challenging syndromes" style="width: 1000px;"/>


Try running the code below on the 10000 shot data set.  See what happens when `osd_method` is set to 1 for OSD-0.  Then try setting this variable to 3 to run a variant of higher order OSD.  Does the logical error rate improve?  How much more time does it take to perform higher order OSD?

In [ ]:
if __name__ == "__main__":
    num_shots = 10000
    run_as_batched = True
    print_output = False
    osd_method = 1  # 0 is off, 1 OSD-0, 3 is OSD-X, where X is the order
    run_decoder(filename, num_shots, run_as_batched, print_output, osd_method)

In [ ]:
if __name__ == "__main__":
    num_shots = 10000
    run_as_batched = True
    print_output = False
    osd_method = 3 # 0 is off, 1 OSD-0, 3 is a higher order OSD
    osd_order = 1
    run_decoder(filename, num_shots, run_as_batched, print_output, osd_method, osd_order)

# Decoder Metrics and (Temporal) Parallel Window Decoding

As QEC matures as a field, research is expanding from its origins in pure theory to experimental demonstrations of QEC workflows on physical QPUs. As QPUs continue to scale, it is becoming increasingly important to consider how every aspect of a QEC workflow scales such that it is possible to run fault-tolerant logic on devices with millions of qubits.

Decoders are likely the primary bottleneck of QEC, so it becomes critically important to understand the nature of these bottlenecks and what sorts of solutions will scale or not.

This notebook will explore the key metrics of decoders, giving you a better intuition for some of the most important practical considerations for decoders and why they matter. It will also explore a scalable decoding scheme called temporal parallel window decoding which has promise to help overcome some of the challenges faced with QEC decoding.


In [ ]:
from Images.parallel.decoder_simulator import QECSimulator

## The Key Decoder Metrics

Recall that the job of a decoder is to take syndromes measured from a quantum computer and a parity check matrix ($H$), defining the parity constraints of the QEC code, and determine where errors occurred so they can be fixed. Decoding a single syndrome extraction round can reveal errors in space, but certain errors, such as measurement errors, can only be caught when decoding in time, using a larger $H$ that can decode multiple rounds of syndrome data.

Decoders can be assessed by three primary metrics:

1. **Accuracy** - The ability to correctly identify errors.
2. **Throughput** - The rate at which the decoder processes syndrome data.
3. **Reaction time** - The time between when the last syndrome is sent from the QPU and the decoder returns a correction.

Accuracy is fairly straightforward. If a decoder makes poor predictions, logical errors occur which results in poor outcomes from the quantum algorithm. This is usually measured by a logical error rate. We will not discuss accuracy much here, but note that there is often a tradeoff between decoder accuracy and its scalability with respect to the number of syndromes decoded in a single block.

Throughput measures how fast syndromes can be processed by the decoder. This rate ($r_{proc}$) must be faster than the rate at which syndromes arrive from the quantum computer $r_{gen}$. If syndromes arrive faster than they are processed, a backlog starts to build up and the decoder grinds to a halt.

Consider an example of the steady-state intercircuit feed-forward latency (SIFL) benchmark shown below. It begins with a preparation of the logical 1 state. Then, 10 syndrome extraction rounds are performed before a measurement occurs and a feed-forward operation determines if an $X$ gate needs to be applied or not before the next measurement.

Between the grey and purple boxes, 10 syndrome extraction rounds occur to flag a bitflip error(s) that might have occurred and potentially induce a logical error when measured. The syndromes must be decoded before the decision is made to apply the $X$ gate or not. This decoding time is depicted as $L_i$ for the $i$th step.

<figure>
  <img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/sifl.png?raw=1" alt="Diagram of the steady-state intercircuit feed-forward latency (SIFL) benchmark circuit showing logical state preparation, repeated syndrome extraction rounds, measurement, and feed-forward X gate application with decoder latency intervals labeled L_i" width="1200">
</figure>

Though this circuit is entirely Clifford, meaning the errors could be tracked and applied at the end, it is a good model for the procedure necessary to apply non-Clifford gates like Toffoli gates or $T$ gates. The main bottleneck of a quantum algorithm is the execution of the non-Clifford gates like $T$ gates. Recall from the lesson ["$T$ Gates and Magic State Distillation"](https://github.com/NVIDIA/cuda-q-academic/blob/master/qec101/05_QEC_MSD.ipynb) that $T$ gates are generally prepared by using an asynchronously generated $\ket{T}$ state via a costly process called **magic state distillation (MSD)**. The $\ket{T}$ states are stored and applied as needed via the teleportation circuit shown below.

<figure>
  <img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/T_teleport.png?raw=1" alt="Circuit diagram of the T gate teleportation protocol showing a magic state input, CNOT gate, measurement, and classically controlled S gate correction" width="600">
</figure>

As demonstrated in the SIFL figure, if $r_{gen}$ < $r_{proc}$, then each step continues with a constant decoder latency $L_i$ (Top). If $r_{gen}$ > $r_{proc}$, a buildup of unprocessed syndromes occurs (bottom) and each subsequent decoder step becomes slower and slower ($L_i$< $L_{i+1}$) before the next $X$ gate is applied.

In Terhal's paper "[Quantum Error Correction for Quantum Memories](https://arxiv.org/pdf/1302.3428)", the argument is made that this backlog becomes a serious problem and grows exponentially with the number of qubits. This means that a decoder with a sufficient throughput is a non-negotiable for QEC.

The final metric "reaction time" essentially sets the limit on the wall clock time for the QPU. It is likely that FTQC circuits will be overwhelmingly limited by application of $T$ or other non-Clifford gates that require feed-forward information from the decoder before proceeding to the next gate.

The reaction time is composed of two main components. First, the latency which is the time from the last syndrome produced by the QPU to the decoder returning its result ($T_{decode}$). The second is the communication overhead from the decoder to the QPU via the classical control system ($T_{latency}$).

$$ T_{reaction}=T_{decode}+T_{latency}$$

Though reaction time is not as critical as throughput, it may be a major practical limitation for completing quantum algorithms with many non-Clifford gates in a reasonable amount of time.

A final honorable mention in the discussion of metrics is scalability. Though, not a metric per se, it is important to remember that any decoder, no matter how promising, needs to scale. Thus, today's research efforts should be directed towards the methods which have the greatest potential to scale and avoid those with proven limitations.


### Exercise 6

Use the `QECSimulator` below to explore the **exponential backlog** problem. The simulator essentially tracks the system load (white line) which consists of all the syndromes currently being decoded and the backlog as a function of wall clock time (of a hypothetical simulated QPU) based on the device parameters you set. The green bars are the time for each decoding round.

To setup the simulation, input the `syndrome_time_us` which is the time in microseconds for a single syndrome extraction from the QPU (this defines $r_{gen}$). The `min_batch_size` specifies the number of syndromes needed to send a batch to the decoder. `decode_func` is a lambda expression which computes how long it will take to decode all $n$ syndromes. This is an important variable to toggle, as the most accurate algorithmic decoders can scale exponentially in terms of the number of syndromes they are decoding. `max_batches_to_run` ensures the simulation stops at a reasonable point. Keep `n_processors` equal to 1 for now.

Run the simulation for the different situations presented below. In the first cell, run the case where the decoder scales linearly with the number of syndromes to decode but is faster than the syndrome generation rate. What happens to the system load and the decoding time?


In [ ]:
# EXERCISE 6

print("Running Scenario 1...")
sim1 = QECSimulator(
    syndrome_time_us=1.0,
    min_batch_size=20,
    decode_func=lambda n: n * 0.8,
    max_batches_to_run=15,
    n_processors=1
)
sim1.run()
sim1.plot_results("Scenario 1: Serial Linear (r_proc > r_gen)")

Now, keep the decode function linear, but change the scaling factor so it is slightly slower than the syndrome generation rate. What happens to the system load and batch decoding times?

In [ ]:
print("Running Scenario 2...")
sim1 = QECSimulator(
    syndrome_time_us=1.0,
    min_batch_size=20,
    decode_func=lambda n: n * 1.2,
    max_batches_to_run=15,
    n_processors=1
)
sim1.run()
sim1.plot_results("Scenario 2: Serial Linear (r_proc < r_gen)")

Now, make the decoder quadratic with the following decode function to better resemble a decoder in practice deployment: $f(n) = 0.01 *n^2$.

In [ ]:
print("Running Scenario 3...")
sim1 = QECSimulator(
    syndrome_time_us=1.0,
    min_batch_size=20,
    decode_func=lambda n: 0.01*n ** 2,
    max_batches_to_run=15,
    n_processors=1
)
sim1.run()
sim1.plot_results("Scenario 3: Serial Quadratic (r_proc > r_gen)")

Even a quadratic scaling decoder can perform well if it is handling a small enough batch of syndromes to keep up, but try changing the prefactor to $f(n) = n^2$ and see what happens.

In [ ]:
print("Running Scenario 4...")
sim1 = QECSimulator(
    syndrome_time_us=1.0,
    min_batch_size=20,
    decode_func=lambda n: n ** 2,
    max_batches_to_run=10,
    n_processors=1
)
sim1.run()
sim1.plot_results("Scenario 4: Serial Quadratic (r_proc << r_gen)")

The simulation can decode the first batch reasonably fast, but the backlog is so severe, the second batch takes 160,000 microseconds! And the simulation continues to blow up after this. Clearly, a more effective method is needed when accurate decoders are required and $r_{proc} < r_{gen}$.

One additional note. The 1 microsecond times for syndrome extraction are in the ballpark for superconducting devices. For slower modalities like ion traps, the same decoder setup may work fine simply because it takes so much more time to perform syndrome extraction. Try running the analysis one more time but using a syndrome extraction time of 200 microseconds.

In [ ]:
print("Running Scenario 5...")
sim1 = QECSimulator(
    syndrome_time_us=200.0,
    min_batch_size=20,
    decode_func=lambda n: n ** 2,
    max_batches_to_run=15,
    n_processors=1
)
sim1.run()
sim1.plot_results("Scenario 5: Serial Quadratic (Ion Trap) (r_proc > r_gen)")

Thus, different decoders may be better suited for different qubit modalities and a whole host of other factors.

## Parallel Window Decoding

The throughput problem is serious, and finding clever ways to increase decoder throughput is necessary to ensure the viability of real-time decoding. Accelerated computing has helped solve simulation problems across all domains of science by massively parallelizing scientific computing and boosting throughput by orders of magnitude. The QEC field is primed to benefit from such an approach as well and this section will build towards a scheme called parallel window decoding which can ameliorate the throughput problem.

Building up from a simple model. If a circuit was entirely composed of Clifford gates, it could be run and all syndromes be decoded after its completion as a single postprocessing step. Unfortunately, the benefits of quantum algorithms comes from non-Clifford gates like $T$ gates which require feedback from a decoder at each application. This means a decoder needs to work alongside the QPU in realtime. The magic states required for application of $T$ gates can also decohere if the decoder backlog is too high. Thus, clever parallel decoding schemes are required to solve the problem.

Let's consider decoding for a single set of 6 stabilizer rounds performed before some $T$ gate. One approach to decoding these could be waiting for all 6 rounds to finish and then decoding them all at once. (The `QECSimulator` above does currently.) This block decoding will be the slowest approach possible as the decoder must process a massive parity (that covers all syndromes) at once.

<figure>
  <img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/alldecode.png?raw=1" alt="Diagram showing block decoding where all 6 syndrome extraction rounds are collected before decoding begins as a single batch" width="500">
</figure>

Aside from the more difficult decoding task, this approach cannot start until all of the syndromes are collected so there is no chance for a headstart.

A second, far more common approach is called **sliding window decoding**. The idea is to decode slices of the syndrome data and feed the results into the next decoding task. One advantage is that the decoding task is much smaller as the "sliding window" covers fewer syndromes at the same time. A second advantage is that the decoder can start working on the first window as soon as it is ready rather than waiting for all of the syndrome data to be generated.

<figure>
  <img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/sliding.png?raw=1" alt="Diagram showing sliding window decoding where overlapping windows of syndrome data are decoded sequentially, with each window covering a subset of rounds" width="500">
</figure>

Even though the decoder gets a head start, the decoding must still occur in serial as each decoding step cannot begin until the previous is finished. Thus, sliding window decoding will quickly run into scaling problems.

The solution to this is **parallel window decoding** as presented in the paper entitled ["Parallel window decoding enables scalable fault tolerant quantum computation"](https://www.nature.com/articles/s41467-023-42482-1). Parallel window decoding can be applied with respect to time (temporal) or space (spatial). For this lesson we will focus on temporal parallelism only, but know that similar techniques could be used to break down a large QEC code patch spatially.

In temporal parallel window decoding, the entire block of syndrome history is obtained and then broken down into sub-blocks that can run in parallel on $N_{proc}$ number of processors in two steps: decoding and cleanup.

<figure>
  <img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/pw_decoding_diagram.png?raw=1" alt="Diagram of parallel window decoding showing syndrome history divided into sub-blocks assigned to multiple processors for parallel decode and cleanup steps" width="500">
</figure>

The decoding step first commits error assignments in specified regions while the second cleans up the boundaries to rectify errors between the commit regions as shown in the image below.

<figure>
  <img src="https://github.com/osbama/Phys437/blob/main/Hands-On/hands_on_3_images/pw_schematic.png?raw=1" alt="Schematic of parallel window decoding showing alternating commit regions (green) and buffer regions, with the two-step decode-then-cleanup procedure illustrated across multiple processors" width="1000">
</figure>

Consider the central commit region in green. There are $n_{com}$ syndromes needed for the decode step. The commit regions are then flanked by buffer regions where syndromes are partially processed, but the results cannot be certain yet. The commit regions are the solved boundaries for the syndromes processed in the second cleanup step. Here, $n_w$ syndromes are processed to determine errors in the two overlapping buffer regions plus any additional syndromes between them.

As you can see from the figure, this pattern repeats for as many processors and sub-blocks as necessary. For counting purposes, the smallest sub-block that can be processed in this manner consists of $n_{com} + n_w$ syndromes. There are many potential choices for the size of these regions. For example, the ["Parallel window decoding enables scalable fault tolerant quantum computation"](https://www.nature.com/articles/s41467-023-42482-1) paper sets $n_{com} = d$ and $n_w = 3d$. Techniques like "[temporal encoding of lattice surgery](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.3.010331)" can allow for a smaller number of syndrome extraction rounds to be used, improving the throughput of the decoder even more by minimizing the sub-block decoding task.

To avoid an exponential backlog if we have a single decoder, the following inequality must hold.

$$ (n_{com} + n_w)t_{\text{syndrome-extraction}} \geq 2 \cdot t_{\text{decode}} $$

At face value this seems like a more challenging decoding task relative to the standard approach as $t_{\text{decode}}$ now has a factor of 2. The trick is that this construction, though requiring two steps, can be attacked with any number of processors $N_{proc}$ so instead the following must hold.

$$ N_{proc}(n_{com} + n_w)t_{\text{syndrome-extraction}} \geq 2 \cdot t_{\text{decode}} $$

This means that a slow decoder using parallel window decoding can still avoid the exponential backlog problem by using an arbitrary number of decoders in parallel up to the limit of the smallest sub-block decoding task. Such an approach has replaced the viability of sliding window and provides a much more promising path to scalable QEC.



### Exercise 7:

Use the QEC Simulator above and determine how many processors are required to keep up with the quadratic scaling decoder you tested earlier. Note that the simulator considers the fact that two decoding rounds are required for parallel window decoding, however the simulator is simplified and divides the syndromes to process into $N$ blocks approximating the procedure to form a buffer zone.



In [ ]:
# EXERCISE 7

print("Running Scenario 6...")
sim1 = QECSimulator(
    syndrome_time_us=1.0,
    min_batch_size=20,
    decode_func=lambda n:  n ** 2,
    max_batches_to_run=15,
    n_processors=7
)
sim1.run()
sim1.plot_results("Scenario 6: Parallel Quadratic (r_proc > r_gen)")

The key takeaway here is that it is often necessary to take a massive decoding task and spend the overhead to distribute it across AI supercomputing resources to avoid the exponential backlog. This is a primary motivation for why GPUs are powerful tools for QEC, even if they might have greater latency than other alternatives.

## Decoder Latency and Reaction Time

Assuming the throughput problem is solved, latency becomes the next critical factor. The latency is the time it takes from the QPU producing the last syndrome to when the decoder obtains a result for this syndrome. This is closely related to a similar quantity called reaction time which also includes the time it takes to transmit the result through the classical control systems.

Rerun your parallel decoding simulation above, but this time use 100 decoders. What happens to the decode time?

In [ ]:
print("Running Scenario 7...")
sim1 = QECSimulator(
    syndrome_time_us=1.0,
    min_batch_size=20,
    decode_func=lambda n:  n ** 2,
    max_batches_to_run=25,
    n_processors=100
)
sim1.run()
sim1.plot_results("Scenario 7: Parallel Quadratic - many processors (r_proc >> r_gen)")

Notice each decoder task is much faster so the result can also be returned to the QPU much faster and progress the quantum computation. Note that more processors can improve the latency, but they are still limited by twice the time it takes to decode a fundamental block. Though this simulator can arbitrarily compute smaller times to give a qualitative depiction of this, in practice, the previous discussion about the size of the buffer and commit regions determines the limit on latency.

A quantum algorithm will be constrained regardless, but there are a number of tradeoffs that can be considered. If the MSD process is slow and few $\ket{T}$ states are on hand, then MSD becomes the bottleneck for QPU computation speed. If the number of $\ket{T}$ states is abundant, then the decoder's reaction time becomes the bottleneck as the decoder must return its result before the next $T$ gate can be applied.

This means that decoding will usually be the primary bottleneck for the QPU wall clock time and might be the difference between an algorithm completing in a reasonable amount of time or not. This is why decoders need to be as fast as possible (even when the throughput problem is solved) and connect to the quantum control devices with interconnects optimized for latency such as [NVIDIA's NVQLink](https://www.nvidia.com/en-us/solutions/quantum-computing/nvqlink/).

Additional space and time tradeoffs can be considered with methods like autocorrected $T$ gates, where $T$ gates proceed without resolving the necessary Clifford correction which is instead tracked as part of the quantum algorithm, speeding up the wall clock time. However, this time savings requires a space cost in the form of additional ancillas for every $T$ gate. Neither approach resolves the underlying constraints but, like many aspects of quantum computing, allows for selection of tradeoffs depending on the application.

## Conclusion

After completing this lab, you should now have a better understanding for why decoding is so difficult and what specific metrics need to be considered when using a decoder in practice. A key takeaway is that the throughput problem is an absolute dealbreaker for decoders. If a decoder cannot process syndromes as fast as they are obtained from the QPU, it will result in an exponential backlog that grinds everything to a halt.

Parallel window decoding is an innovative solution to this problem that allows many processors working in tandem to solve the throughput problem.

Response time and accuracy are also very important and might impose severe practical limitations on a QEC workflow, but these are secondary to throughput.